# GDL100 - 기하 딥러닝(Geometric Deep Learning) - 실습 3

GDL100 - 기하 딥러닝의 **실습 3(Practical 3)**에 오신 것을 진심으로 환영합니다!

## 소개

그래프 신경망(Graph Neural Networks, GNN)은 **기하 딥러닝(Geometric Deep Learning)**이라고 불리는 폭넓고 새롭게 떠오르는 연구 패러다임의 일부입니다. 이는 데이터에 내재된 불변성(invariance)과 대칭성(symmetry)을 존중하는 신경망 아키텍처를 고안하는 것을 의미합니다. 이번 실습은 기하 딥러닝의 세계로 부드럽게 입문하는 것을 목표로 합니다.

이번 실습의 **목표**는 다음과 같습니다:

* 그래프 신경망의 **불변(invariant)** 및 **등변(equivariant)** 성질을 이론과 증명에서부터 프로그래밍 및 단위 테스트(unit testing)에 이르기까지 이해하기.
* 최첨단 GNN 및 기하 딥러닝 모델을 개발하는 데 널리 쓰이는 라이브러리인 [**PyTorch Geometric**](https://pytorch-geometric.readthedocs.io/en/latest/)(PyG)을 직접 다뤄보기. 특히, 새로운 GNN 레이어를 설계하기 위한 `MessagePassing` 기반 클래스와 그래프 데이터셋을 표현하기 위한 `Data` 객체에 익숙해지기.
* **3D 공간**에 위치한 그래프 데이터의 **기하 정보(geometric information)**를 활용하는 GNN 레이어를 구성하는 근본 원리를 음미하기. 이는 GNN 연구에서 매우 활발한 분야입니다.

## 지금까지 다룬 내용

실습 1과 강의를 통해 우리는 다음 내용을 다뤘습니다:
* 그래프 데이터에 대한 노드 수준 예측 작업을 위한 그래프 합성곱 신경망(Graph Convolutional Networks) 소개. 예: Cora 그래프에서 연구 논문 분류하기.
* 그래프 수준 예측을 수행하는 방법. 예: ZINC에서 분자 특성 예측 (참고: 메시지 패싱 레이어 다음에 전역 풀링(global pooling)을 적용).
* 샘플마다 그래프 크기가 가변적인 그래프 데이터셋을 배칭(batching)할 때의 어려움.
* 그래프 구조의 밀집 인접 행렬($|V| \times |V|$) 표현과 희소 엣지 인덱스($2 \times |E|$) 표현을 비교하기 (좋은 복습 자료는 [여기](https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html#mini-batche)에 있습니다).
* 서로 다른 GNN들의 표현력(expressive power)을 특성화하기.

이번 실습에서는 특정한 구조적 규칙성(structural regularities)이 존재할 때 강력한 그래프 신경망을 개발하는 방법을 공부할 것입니다.


## 면책 조항

이번 실습은 **도전적**이고 **사고를 확장시키도록** 의도되었습니다.

이것이 석사 수준의 과정이므로, 우리는 정답에 이르는 경로가 결코 잘 정의되어 있거나 명백하지 않은 실제 연구를 여러분이 준비할 수 있도록 노력하고 있습니다. 노트북 전반에 걸쳐 여러분의 사고 과정을 설명할 것을 적극 권장합니다. 채점자의 관점에서 볼 때, 최상위 학생은 문제를 이해했음을 보여주고 현재의 GNN 지식을 바탕으로 원칙에 입각한 해결책을 설계하려고 시도한 학생일 것입니다.

## 저자

**저자는 다음과 같습니다**: 질문이나 피드백이 있으면 주저하지 말고 저희(또는 다른 TA들)에게 연락해 주세요!

- Chaitanya K. Joshi (ckj24@cl.cam.ac.uk)
- Charlie Harris (cch57@cam.ac.uk)
- Ramon Viñas Torné (rv340@cam.ac.uk)

---
---
---

# ⚙️ Part 0: 설치 및 설정

**❗️참고:** 이 실습을 완료하려면 GPU가 필요합니다. `Runtime -> Change runtime type`을 클릭하고 `hardware accelerator`를 **GPU**로 설정하는 것을 잊지 마세요.

In [ ]:
#@title [RUN] Install required python libraries
import os

# Install PyTorch Geometric and other libraries
if 'IS_GRADESCOPE_ENV' not in os.environ:
    !pip install -q torch-scatter -f https://pytorch-geometric.com/whl/torch-1.10.0+cu111.html
    !pip install -q torch-sparse -f https://pytorch-geometric.com/whl/torch-1.10.0+cu111.html
    !pip install -q torch-geometric==2.0.3
    !pip install -q rdkit-pypi==2021.9.4
    !pip install -q py3Dmol==1.8.0


In [ ]:
#@title [RUN] Import python modules

import os
import time
import random
import numpy as np

from scipy.stats import ortho_group

import torch
import torch.nn.functional as F
from torch.nn import Linear, ReLU, BatchNorm1d, Module, Sequential

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.data import Batch
from torch_geometric.datasets import QM9
import torch_geometric.transforms as T
from torch_geometric.utils import remove_self_loops, to_dense_adj, dense_to_sparse
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.datasets import QM9
from torch_scatter import scatter

import rdkit.Chem as Chem
from rdkit.Geometry.rdGeometry import Point3D
from rdkit.Chem import QED, Crippen, rdMolDescriptors, rdmolops
from rdkit.Chem.Draw import IPythonConsole

import py3Dmol
from rdkit.Chem import AllChem

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from google.colab import files
from IPython.display import HTML

print("PyTorch version {}".format(torch.__version__))
print("PyG version {}".format(torch_geometric.__version__))

In [ ]:
#@title [RUN] Set random seed for deterministic results

def seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed(0)

In [ ]:
#@title [RUN] Helper functions for data preparation

class SetTarget(object):
    """
    이 변환(transform)은 각 데이터 샘플의 레이블 벡터를 수정하여
    특정 타깃에 대한 레이블만 남깁니다 (QM9에는 19개의 타깃이 있습니다).

    참고: 이번 실습에서는 타깃을 0번 타깃으로 하드코딩했습니다.
    즉, 약물 유사 분자(drug-like molecule)의 전기 쌍극자 모멘트(electric dipole moment)입니다.
    (https://en.wikipedia.org/wiki/Electric_dipole_moment)
    """
    def __call__(self, data):
        target = 0 # we hardcoded choice of target
        data.y = data.y[:, target]
        return data


class CompleteGraph(object):
    """
    이 변환(transform)은 각 데이터 샘플의 엣지 인덱스에 모든 쌍별(pairwise) 엣지를 추가한 뒤
    셀프 루프(self loop)를 제거합니다. 즉, 완전 연결(fully connected) 또는 완전(complete) 그래프를 만듭니다.
    """
    def __call__(self, data):
        device = data.edge_index.device

        row = torch.arange(data.num_nodes, dtype=torch.long, device=device)
        col = torch.arange(data.num_nodes, dtype=torch.long, device=device)

        row = row.view(-1, 1).repeat(1, data.num_nodes).view(-1)
        col = col.repeat(data.num_nodes)
        edge_index = torch.stack([row, col], dim=0)

        edge_attr = None
        if data.edge_attr is not None:
            idx = data.edge_index[0] * data.num_nodes + data.edge_index[1]
            size = list(data.edge_attr.size())
            size[0] = data.num_nodes * data.num_nodes
            edge_attr = data.edge_attr.new_zeros(size)
            edge_attr[idx] = data.edge_attr

        edge_index, edge_attr = remove_self_loops(edge_index, edge_attr)
        data.edge_attr = edge_attr
        data.edge_index = edge_index

        return data

In [ ]:
#@title [RUN] Helper functions for visualization

allowable_atoms = [
    "H",
    "C",
    "N",
    "O",
    "F",
    "C",
    "Cl",
    "Br",
    "I",
    "H",
    "Unknown",
]

def to_atom(t):
    try:
        return allowable_atoms[int(t.argmax())]
    except:
        return "C"


def to_bond_index(t):
    t_s = t.squeeze()
    return [1, 2, 3, 4][
        int(
            torch.dot(
                t_s,
                torch.tensor(
                    range(t_s.size()[0]), dtype=torch.float, device=t.device
                ),
            ).item()
        )
    ]

def to_rdkit(data, device=None):
    has_pos = False
    node_list = []
    for i in range(data.x.size()[0]):
        node_list.append(to_atom(data.x[i][:5]))

    # 편집 가능한 빈 mol 객체 생성
    mol = Chem.RWMol()
    # mol에 원자를 추가하고 인덱스를 추적
    node_to_idx = {}
    invalid_idx = set([])
    for i in range(len(node_list)):
        if node_list[i] == "Stop" or node_list[i] == "H":
            invalid_idx.add(i)
            continue
        a = Chem.Atom(node_list[i])
        molIdx = mol.AddAtom(a)
        node_to_idx[i] = molIdx

    added_bonds = set([])
    for i in range(0, data.edge_index.size()[1]):
        ix = data.edge_index[0][i].item()
        iy = data.edge_index[1][i].item()
        bond = to_bond_index(data.edge_attr[i])  # <font color='red'>TODO</font> fix this
        # bond = 1
        # 인접한 원자들 사이에 결합(bond) 추가

        if data.edge_attr[i].sum() == 0:
          continue

        if (
            (str((ix, iy)) in added_bonds)
            or (str((iy, ix)) in added_bonds)
            or (iy in invalid_idx or ix in invalid_idx)
        ):
            continue
        # 해당하는 결합 유형 추가 (이외에도 훨씬 더 많은 종류가 있습니다)

        if bond == 0:
            continue
        elif bond == 1:
            bond_type = Chem.rdchem.BondType.SINGLE
            mol.AddBond(node_to_idx[ix], node_to_idx[iy], bond_type)
        elif bond == 2:
            bond_type = Chem.rdchem.BondType.DOUBLE
            mol.AddBond(node_to_idx[ix], node_to_idx[iy], bond_type)
        elif bond == 3:
            bond_type = Chem.rdchem.BondType.TRIPLE
            mol.AddBond(node_to_idx[ix], node_to_idx[iy], bond_type)
        elif bond == 4:
            bond_type = Chem.rdchem.BondType.SINGLE
            mol.AddBond(node_to_idx[ix], node_to_idx[iy], bond_type)

        added_bonds.add(str((ix, iy)))

    if has_pos:
        conf = Chem.Conformer(mol.GetNumAtoms())
        for i in range(data.pos.size(0)):
            if i in invalid_idx:
                continue
            p = Point3D(
                data.pos[i][0].item(),
                data.pos[i][1].item(),
                data.pos[i][2].item(),
            )
            conf.SetAtomPosition(node_to_idx[i], p)
        conf.SetId(0)
        mol.AddConformer(conf)

    # RWMol을 Mol 객체로 변환
    mol = mol.GetMol()
    mol_frags = rdmolops.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    largest_mol = max(mol_frags, default=mol, key=lambda m: m.GetNumAtoms())
    return largest_mol


def MolTo3DView(mol, size=(300, 300), style="stick", surface=False, opacity=0.5):
    """분자를 3D로 그립니다

    Args:
    ----
        mol: rdMol, 표시할 분자
        size: tuple(int, int), 캔버스 크기
        style: str, 분자를 그리는 방식
               style은 'line', 'stick', 'sphere', 'carton'이 가능합니다
        surface, bool, SAS 표시 여부
        opacity, float, surface의 불투명도, 범위 0.0-1.0
    Return:
    ----
        viewer: py3Dmol.view, ipython 노트북에 임베드된 3Dmol.js 뷰를 구성하기 위한 클래스.
    """
    assert style in ('line', 'stick', 'sphere', 'carton')

    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view(width=size[0], height=size[1])
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({style:{}})
    if surface:
        viewer.addSurface(py3Dmol.SAS, {'opacity': opacity})
    viewer.zoomTo()
    return viewer

def smi2conf(smiles):
    '''SMILES를 3D 좌표를 가진 rdkit.Mol로 변환합니다'''
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol)
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
        return mol
    else:
        return None

In [ ]:
# 실습 과정에서 실험 결과를 저장하기 위한 용도
RESULTS = {}
DF_RESULTS = pd.DataFrame(columns=["Test MAE", "Val MAE", "Epoch", "Model"])

좋습니다! 이제 실습 3에 본격적으로 뛰어들 준비가 되었습니다!

---
---
---

# 🧪 Part 0: PyTorch Geometric을 활용한 분자 특성 예측 소개

이 섹션에서는 기초를 다룹니다. 우리는 그래프 신경망(GNN)이 분자의 화학적 특성을 예측하는 데 어떻게 활용될 수 있는지 공부할 것입니다. 이는 기하 딥러닝의 영향력 있는 실세계 응용 사례입니다. 이를 위해 먼저 GNN 구현을 용이하게 하는 널리 쓰이는 파이썬 라이브러리인 PyTorch Geometric을 소개하겠습니다.

## PyTorch Geometric

[PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/)(PyG)은 그래프 표현 학습(graph representation learning) 연구 및 개발을 위한 훌륭한 라이브러리입니다:

> PyTorch Geometric(PyG)은 다양한 출판된 논문에서 가져온, 그래프 및 기타 불규칙 구조(irregular structures)에 대한 딥러닝 방법들(기하 딥러닝이라고도 함)로 구성되어 있습니다. 또한 다수의 작은 그래프나 하나의 거대한 그래프를 다루기 위한 사용하기 쉬운 미니배치 로더, 멀티 GPU 지원, 분산 그래프 학습, 다수의 일반적인 벤치마크 데이터셋, 그리고 임의의 그래프뿐 아니라 3D 메시(mesh)나 점군(point cloud)에 대한 학습 모두에 유용한 변환(transform)들을 제공합니다.

이번 실습에서는 PyG를 광범위하게 사용할 것입니다. PyG를 한 번도 다뤄본 적이 없더라도 걱정하지 마세요. 몇 가지 예제를 제공하고 모든 기초를 자세하게 안내해 드리겠습니다. 또한 [이 자기 완결형 공식 튜토리얼](https://pytorch-geometric.readthedocs.io/en/latest/notes/introduction.html)을 적극 추천하는데, 이것이 여러분이 시작하는 데 도움이 될 것입니다. 무엇보다도, 일반적인 PyG [Message Passing](https://pytorch-geometric.readthedocs.io/en/latest/notes/create_gnn.html) 클래스를 통해 최첨단 GNN 레이어를 구현하는 방법을 배우게 됩니다 (이에 대해서는 나중에 더 다룹니다).

이제 분자 특성을 예측하는 문제로 관심을 돌려봅시다.

## 분자 특성 예측 (Molecular Property Prediction)

분자는 결합(엣지)으로 연결된 원자(노드)의 그래프로 쉽게 표현할 수 있는, 자연에서 나온 객체의 훌륭한 예시입니다.
화학에서 GNN의 인기 있는 응용 분야는 **분자 특성 예측(Molecular Property Prediction)** 작업입니다. 목표는 약물 유사 분자(drug-like molecule)의 유용한 특성을 예측할 수 있는 GNN 모델을 과거 실험 데이터로부터 학습시키는 것입니다. 그러면 모델의 예측은 신약 설계 과정을 안내하는 데 사용될 수 있습니다.

<!-- ![](https://drive.google.com/uc?id=1Hs6fMSZ6a0WdjKqzbmBME0RYoSwxMaYQ) -->
<img src="https://github.com/chaitjo/dump/raw/main/molproppred.png">

분자 특성 예측에 GNN이 사용된 유명한 예시 중 하나는 **항생제 발견(antibiotic discovery)**의 세계에 있습니다. 이는 인류에게 잠재적으로 막대한 영향을 미칠 수 있지만 악명 높게도 혁신이 거의 없었던 분야입니다. 어떤 분자가 박테리아를 얼마나 억제할지 예측하도록 훈련된 GNN은 가상 스크리닝(virtual screening) 과정에서 이전에 간과되었던 화합물 [**할리신(Halicin)**](https://www.wikiwand.com/en/Halicin)(아래)을 식별해낼 수 있었습니다. 할리신은 *인 비트로(in vitro)*(세포 내) 시험에서 강력한 결과를 보였을 뿐만 아니라, 어떤 박테리아도 (아직까지) 내성을 발전시키지 못한 완전히 새로운 작용 기전(mechanism of action)을 가지고 있었습니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Halicin.svg/440px-Halicin.svg.png" width="30%">

## QM9 데이터셋

QM9(Quantum Mechanics dataset 9)는 19개의 회귀(regression) 타깃을 가진 약 **130,000개의 작은 분자**로 구성된 데이터셋입니다. [MoleculeNet](https://arxiv.org/abs/1703.00564)에서 사용된 이후로, 분자 특성 예측을 위한 새로운 아키텍처를 벤치마킹하는 인기 있는 데이터셋이 되었습니다.

구체적으로, 우리는 약물 유사 분자의 [전기 쌍극자 모멘트(electric dipole moment)](https://en.wikipedia.org/wiki/Electric_dipole_moment)를 예측할 것입니다. 위키피디아에 따르면:
> "전기 쌍극자 모멘트는 한 시스템 내에서 양전하와 음전하의 분리 정도를 나타내는 척도, 즉 시스템의 전체적인 극성(polarity)을 나타내는 척도이다."

우리는 이 개념을 물 분자 H<sub>2</sub>0를 통해 시각화할 수 있습니다. 물 분자는 음전하(파란색)와 양전하(빨간색)의 분포가 약간 다르기 때문에 쌍극자를 형성합니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Water-elpot-transparent-3D-balls.png/500px-Water-elpot-transparent-3D-balls.png" width="25%">

쌍극자 모멘트의 바탕이 되는 정확한 물리적·화학적 원리에 대해서는 걱정할 필요가 없습니다. 짐작할 수 있듯이, 이러한 특성을 예측하기 위해 제1원리(first principles)로부터 방정식을 쓰는 것은, 특히 복잡한 분자(예: 단백질)의 경우 매우 어렵습니다. (적어도 이번 실습에서는) 여러분이 알아야 할 것은, 이러한 분자들이 노드 및 엣지 특성과 더불어 **공간 정보(spatial information)**를 가진 그래프로 표현될 수 있으며, 이를 활용해 정답 레이블을 이용하여 GNN 모델을 훈련시킬 수 있다는 것뿐입니다.

이제 QM9 데이터셋을 불러와 분자 그래프가 어떻게 표현되는지 탐색해 봅시다. PyG는 이것을 매우 편리하게 만들어 줍니다.

(데이터셋을 다운로드하는 데 몇 분 정도 걸릴 수 있습니다.)

In [ ]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    path = './qm9'
    target = 0

    # 데이터 로딩 중에 적용되는 변환(transform):
    # (1) 그래프를 완전 연결, (2) 타깃/레이블 선택
    transform = T.Compose([CompleteGraph(), SetTarget()])

    # 정의된 변환과 함께 QM9 데이터셋 로드
    dataset = QM9(path, transform=transform)

    # 각 데이터 샘플의 타깃을 평균 = 0, 표준편차 = 1로 정규화.
    mean = dataset.data.y.mean(dim=0, keepdim=True)
    std = dataset.data.y.std(dim=0, keepdim=True)
    dataset.data.y = (dataset.data.y - mean) / std
    mean, std = mean[:, target].item(), std[:, target].item()

## 데이터 준비 및 분할

QM9 데이터셋에는 **130,000개**가 넘는 분자 그래프가 있습니다!

이번 실습의 목적을 위해, 보다 다루기 쉬운 **3,000개**의 분자 그래프 부분집합(sub-set)을 만들고 이를 훈련(training), 검증(validation), 테스트(test) 세트로 나눠봅시다. 훈련, 검증, 테스트 각각에 1,000개의 그래프를 사용할 것입니다.

이번 실습의 후반부에서는 QM9 데이터셋의 전체/더 큰 부분집합으로도 실험해 볼 수 있습니다.

In [ ]:
print(f"Total number of samples: {len(dataset)}.")

# 데이터셋 분할 (전체 데이터셋을 사용하는 경우)
# test_dataset = dataset[:10000]
# val_dataset = dataset[10000:20000]
# train_dataset = dataset[20000:]

# 데이터셋 분할 (우리의 3K 부분집합)
train_dataset = dataset[:1000]
val_dataset = dataset[1000:2000]
test_dataset = dataset[2000:3000]
print(f"Created dataset splits with {len(train_dataset)} training, {len(val_dataset)} validation, {len(test_dataset)} test samples.")

# 배치 크기 = 32로 데이터로더 생성
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 분자 그래프 시각화

QM9 분자 그래프가 어떻게 생겼는지 더 잘 이해하기 위해, 훈련 세트의 몇몇 샘플을 해당하는 타깃(쌍극자 모멘트)과 함께 시각화해 봅시다.

다음 플롯에서는 엣지가 물리적 연결(즉, 결합)을 나타내는 **희소 그래프(sparse graphs)**를 시각화합니다. 그러나 이번 실습에서는 **완전 연결 그래프(fully-connected graphs)**를 사용하고 그래프 구조를 각 엣지의 속성(attribute)에 인코딩할 것입니다. 이번 실습 후반부에서 두 접근법의 장점과 단점을 모두 공부할 것입니다.

**❗️참고:** 우리는 PyG 그래프를 화학 및 분자 시각화를 위한 파이썬 패키지인 RDKit에서 사용할 수 있는 Molecule 객체로 변환하는 코드를 여러분을 위해 일부 구현해 두었습니다. 시각화 목적 이상으로 RDKit을 이해하는 것은 중요하지 않습니다.

In [ ]:
num_viz = 50
mols = [to_rdkit(train_dataset[i]) for i in range(num_viz)]
values = [str(round(float(train_dataset[i].y), 3)) for i in range(num_viz)]

Chem.Draw.MolsToGridImage(mols, legends=[f"y = {value}" for value in values], molsPerRow=5)

## PyG Data 객체 이해하기

우리 데이터셋의 각 그래프는 PyG `Data` 객체로 캡슐화되어 있습니다. 이는 기하 딥러닝에서 사용하기 위한 모든 구조화된 데이터(그래프, 점군, 메시 포함)를 표현하는 편리한 방법입니다.

In [ ]:
data = train_dataset[0] # one data sample, i.e. molecular graph
print("Let us print all the attributes (along with their shapes) that our PyG molecular graph contains:")
print(data)

`Data` 객체의 인스턴스 안에서, 개별 `Torch.Tensor` 속성(또는 다른 어떤 변수 유형이든)은 신경망 레이어 내에서 점(dot)으로 쉽게 접근할 수 있습니다. PyG에서 가져온 그래프에는 미리 계산된 여러 특성이 함께 제공되며, 이를 아래에 설명합니다 (여기 나오는 화학 용어에 익숙하지 않더라도 걱정하지 마세요):

**원자 특성 (`data.x`)** - $\mathbb{R}^{|V| \times 11}$
- 1~5번째 특성: 원자 유형 (원-핫: H, C, N, O, F)
- 6번째 특성 (`data.z`이기도 함): 원자 번호(양성자의 수).
- 7번째 특성: 방향족(aromatic) 여부 (이진)
- 8~10번째 특성: 전자 궤도 혼성화(orbital hybridization) (원-핫: sp, sp2, sp3)
- 11번째 특성: 수소의 개수

**엣지 인덱스 (`data.edge_index`)** - $\mathbb{R}^{2×|E|}$
- 그래프의 엣지 연결성을 설명하는 2 x `num_edges` 차원의 텐서

**엣지 특성 (`data.edge_attr`)** - $\mathbb{R}^{|E|\times 4}$
- 1~4번째 특성: 결합 유형 (원-핫: 단일, 이중, 삼중, 방향족)

**원자 위치 (`data.pos`)** - $\mathbb{R}^{|V|\times 3}$
- 각 원자의 3D 좌표. (이것의 중요성에 대해서는 실습 후반부에서 다룰 것입니다.)

**타깃 (`data.y`)** - $\mathbb{R}^{1}$
- 분자의 전기 쌍극자 모멘트에 해당하는 스칼라 값

**❗️참고:** 우리는 **완전 연결 그래프(fully-connected graphs)**를 사용할 것입니다 (즉, 분자 내 모든 원자가 셀프 루프를 제외하고 서로 연결됨). 분자 구조에 관한 정보는 다음과 같이 엣지 특성(`data.edge_attr`)을 통해 모델에 제공됩니다:
- 두 원자가 물리적으로 연결되어 있을 때, 엣지 속성은 원-핫 벡터를 통해 **결합 유형**(단일, 이중, 삼중, 또는 방향족)을 나타냅니다.
- 두 원자가 물리적으로 연결되어 있지 않을 때, **모든 엣지 속성**은 **0**입니다.
우리는 나중에 완전 연결 인접 행렬과 희소 인접 행렬(두 원자 사이에 물리적 연결이 존재할 때만 엣지가 존재함)의 장점/단점을 공부할 것입니다.

In [ ]:
print(f"\nThis molecule has {data.x.shape[0]} atoms, and {data.edge_attr.shape[0]} edges.")

print(f"\nFor each atom, we are given a feature vector with {data.x.shape[1]} entries (described above).")

print(f"\nFor each edge, we are given a feature vector with {data.edge_attr.shape[1]} entries (also described above).")

print(f"\nIn the next section, we will learn how to build a GNN in the Message Passing flavor to process the node and edge features of molecular graphs and predict their properties.")

print(f"\nEach atom also has a {data.pos.shape[1]}-dimensional coordinate associated with it. We will talk about their importance later in the practical.")

print(f"\nFinally, we have {data.y.shape[0]} regression target for the entire molecule.")

## 배칭(batching)을 위한 PyG 사용

이전 실습에서 기억할 수 있듯이, **그래프 배칭(batching)**은 꽤 번거롭고 까다로운 과정일 수 있습니다. 다행히도 PyG를 사용하면 이것이 아주 간단해집니다! `Data` 객체들의 리스트가 주어지면, 이를 손쉽게 PyG `Batch` 객체로 배칭할 수 있을 뿐만 아니라 다시 그래프들의 리스트로 언배칭(unbatch)할 수도 있습니다. 게다가, 우리 경우처럼 간단한 상황에서는 PyG `DataLoader` 객체(기본 PyTorch의 것과는 다름)가 내부적으로 모든 배칭을 처리해 줍니다!

그래도 시연을 위해 몇몇 그래프를 빠르게 배칭하고 언배칭해 봅시다:

In [ ]:
# Toy graph 1
edge_index_1 = torch.tensor(
    [[0, 1, 1, 2], [1, 0, 2, 1]],
    dtype=torch.long
)
x_1 = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data_1 = Data(x=x_1, edge_index=edge_index_1)

# Toy graph 2
edge_index_2 = torch.tensor(
    [[0, 2, 1, 0], [2, 0, 0, 1]],
    dtype=torch.long
)
x_2 = torch.tensor([[1], [0], [-1]], dtype=torch.float)

data_2 = Data(x=x_2, edge_index=edge_index_2)

# 토이 그래프들로 배치 생성
data_list = [data_1, data_2]
batch = Batch.from_data_list(data_list)

assert (batch[0].x == data_1.x).all() and (batch[1].x == data_2.x).all()

# DataLoader 생성
loader = DataLoader(data_list, batch_size=1, shuffle=False)
it = iter(loader)
batch_1 = next(it)
batch_2 = next(it)

assert (batch_1.x == data_1.x).all() and (batch_2.x == data_2.x).all()

훌륭합니다! 우리는 QM9 데이터셋을 다운로드하고 준비했으며, 몇몇 샘플을 시각화하고, 각 분자 그래프와 연관된 속성을 이해했으며, PyG에서 배칭이 어떻게 작동하는지 검토했습니다. 이제 분자 특성 예측을 위해 PyG에서 GNN을 어떻게 개발할 수 있는지 이해할 준비가 되었습니다.

---
---
---

# 📩 Part 0: PyTorch Geometric에서의 메시지 패싱 신경망(Message Passing Neural Networks) 소개

PyTorch Geometric에 대한 부드러운 입문으로서, **메시지 패싱(Message Passing)** 방식으로 GNN을 개발하는 첫걸음을 함께 살펴보겠습니다.

<!-- ![](https://drive.google.com/uc?id=1Wdgdq606XW1MelvcU1nW5CxWe1rHsWt1) -->
<img src="https://github.com/chaitjo/dump/raw/main/gnn-layers.png">

## 형식화 (Formalism)

먼저, 우리의 분자 특성 예측 파이프라인을 형식화해 봅시다. (우리의 표기법은 대부분 강의에서 소개된 것을 따르지만, 변수 이름에 대해서는 몇 가지 다른 선택을 합니다.)

### 그래프
분자 그래프 $\mathcal{G} = \left( \mathcal{V}, \mathcal{E} \right)$를 생각해 봅시다. 여기서 $\mathcal{V}$는 $n$개 노드의 집합이고, $\mathcal{E}$는 노드들과 연관된 엣지의 집합입니다. 각 노드 $i \in \mathcal{V}$에 대해, $d_n$차원의 초기 특성 벡터 $h_i \in \mathbb{R}^{d_n}$가 주어집니다.
각 엣지 $(i, j) \in \mathcal{E}$에 대해, $d_e$차원의 초기 특성 벡터 $e_{ij} \in \mathbb{R}^{d_e}$가 주어집니다. QM9 그래프의 경우, $d_n = 11, d_e = 4$입니다.

### 레이블/타깃
각 그래프 $\mathcal{G}$에는 우리가 예측하고자 하는 스칼라 타깃 또는 레이블 $y \in \mathbb{R}^{1}$이 연관되어 있습니다.

이를 위해 우리는 그래프 특성 예측을 위한 메시지 패싱 신경망(Message Passing Neural Network)을 설계할 것입니다. 우리의 MPNN은 여러 층의 메시지 패싱과, 그 뒤를 잇는 전역 풀링(global pooling) 및 예측 헤드(prediction head)로 구성됩니다.

### MPNN 레이어
메시지 패싱 연산은 다음 방정식을 통해 노드 특성 $h_i^{\ell} \in \mathbb{R}^d$를 층 $\ell$에서 층 $\ell+1$로 반복적으로 갱신합니다:
$$
h_i^{\ell+1} = \phi \Bigg( h_i^{\ell}, \oplus_{j \in \mathcal{N}_i} \Big( \psi \left( h_i^{\ell}, h_j^{\ell}, e_{ij} \right) \Big) \Bigg),
$$
여기서 $\psi, \phi$는 다층 퍼셉트론(Multi-Layer Perceptrons, MLP)이고, $\oplus$는 합산(summation), 최댓값(maximization), 또는 평균(averaging)과 같은 순열 불변(permutation-invariant) 국소 이웃 집계 함수(local neighborhood aggregation function)입니다.

MPNN 레이어를 교육적인 세 단계로 나눠봅시다:
- **단계 (1): 메시지(Message).** 연결된 각 노드 쌍 $i, j$에 대해, 네트워크는 먼저 메시지 $m_{ij} =  \psi \left( h_i^{\ell}, h_j^{\ell}, e_{ij} \right)$를 계산합니다. MLP $\psi: \mathbb{R}^{2d + d_e} → \mathbb{R}^d$는 출발 노드(source node), 도착 노드(destination node), 엣지의 특성 벡터를 연결(concatenation)한 것을 입력으로 받습니다.
    - 첫 번째 층 $\ell=0$에서는 $h_i^{\ell=0} = W_{in} \left( h_i \right)$임에 유의하세요. 여기서 $W_{in} \in \mathbb{R}^{d_n}  \rightarrow \mathbb{R}^{d}$는 초기 노드 특성을 은닉 차원 $d$로 보내는 단순한 선형 사영(linear projection, `torch.nn.Linear`)입니다.
- **단계 (2): 집계(Aggregate).** 각 노드 $i$에서, 모든 이웃으로부터 들어오는 메시지가 $m_{i} = \oplus_{{j \in \mathcal{N}_i}} \left( m_{ij} \right)$로 집계됩니다. 여기서 $\oplus$는 순열 불변 함수입니다. 우리는 합산을 사용할 것입니다. 즉, $\oplus_{{j \in \mathcal{N}_i}} = \sum_{{j \in \mathcal{N}_i}}$입니다.
- **단계 (3): 갱신(Update).** 마지막으로, 네트워크는 집계된 메시지 $m_i$와 이전 노드 특성 벡터 $h_i^{\ell}$를 연결(concatenate)하여 MLP $\phi: \mathbb{R}^{2d} → \mathbb{R}^{d}$에 통과시킴으로써 노드 특성 벡터 $h_i^{\ell+1} = \phi \left( h_i^{\ell}, m_i \right)$를 갱신합니다.

### 전역 풀링 및 예측 헤드
$L$개 층의 메시지 패싱 후, 우리는 최종 노드 특성 $h_i^{\ell=L}$을 얻습니다. 그래프당 하나의 타깃 $y$가 있으므로, 우리는 모든 노드 특성을 하나의 그래프 특성 또는 그래프 임베딩 $h_G \in \mathbb{R}^d$로 풀링해야 합니다. 이는 때때로 '리드아웃(readout)' 함수라고 불리는 또 다른 순열 불변 함수 $R$을 통해 다음과 같이 이뤄집니다:
$$
h_G = R_{i \in \mathcal{V}} \left( h_i^{\ell=L} \right).
$$
우리는 모든 노드 특성에 대한 전역 평균 풀링(global average pooling)을 사용할 것입니다. 즉,
$$
h_G = \frac{1}{|\mathcal{V}|} \sum_{i \in \mathcal{V}} h_i^{\ell=L}.
$$

그래프 임베딩 $h_G$는 선형 예측 헤드 $W_{pred} \in \mathbb{R}^{d} \rightarrow \mathbb{R}^1$를 통과하여 전체 예측값 $\hat y \in \mathbb{R}^1$을 얻습니다:
$$
\hat y = W_{pred} \left( h_G \right).
$$

### 손실 함수
우리의 MPNN 그래프 특성 예측 모델은 회귀를 위한 표준 평균 제곱 오차(mean-squared error) 손실을 최소화함으로써 종단간(end-to-end)으로 훈련될 수 있습니다:
$$
\mathcal{L}_{MSE} = \lVert y - \hat y \rVert^2_2.
$$

## 기본 메시지 패싱 신경망(Message Passing Neural Network) 레이어 코딩하기

이제 위에서 설명한 내용을 구현하는 기본 MPNN 레이어를 정의할 준비가 되었습니다. 특히, 먼저 **MPNN 레이어**를 코딩할 것입니다. (나머지 부분은 이후에 코딩하겠습니다.)

이를 위해 `MessagePassing` 베이스 클래스를 상속받을 것인데, 이 클래스는 메시지 전파(message propagation)를 자동으로 처리해 주며 고급 GNN 모델을 개발하는 데 매우 유용합니다. 커스텀 MPNN을 구현하려면, 사용자는 `message`(즉 $\psi$), `aggregate`(즉 $\oplus$), `update`(즉 $\phi$) 함수의 동작만 정의하면 됩니다. 커스텀 메시지 패싱 레이어 구현에 관해서는 [PyG 문서](https://pytorch-geometric.readthedocs.io/en/latest/notes/create_gnn.html)도 참고할 수 있습니다.

아래에서는 무슨 일이 일어나고 있는지 파악하는 데 도움이 되도록 풍부한 인라인 주석과 함께 표준 MPNN 레이어의 구현을 예시로 제공합니다.

In [ ]:
class MPNNLayer(MessagePassing):
    def __init__(self, emb_dim=64, edge_dim=4, aggr='add'):
        """메시지 패싱 신경망(Message Passing Neural Network) 레이어

        Args:
            emb_dim: (int) - 은닉 차원 `d`
            edge_dim: (int) - 엣지 특징(edge feature) 차원 `d_e`
            aggr: (str) - 집계 함수 `\oplus` (sum/mean/max)
        """
        # 집계 함수를 설정
        super().__init__(aggr=aggr)

        self.emb_dim = emb_dim
        self.edge_dim = edge_dim

        # 메시지 `m_ij`를 계산하기 위한 MLP `\psi`
        # Linear->BN->ReLU->Linear->BN->ReLU 스택으로 구현됨
        # 차원: (2d + d_e) -> d
        self.mlp_msg = Sequential(
            Linear(2*emb_dim + edge_dim, emb_dim), BatchNorm1d(emb_dim), ReLU(),
            Linear(emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU()
          )

        # 업데이트된 노드 특징 `h_i^{l+1}`를 계산하기 위한 MLP `\phi`
        # Linear->BN->ReLU->Linear->BN->ReLU 스택으로 구현됨
        # 차원: 2d -> d
        self.mlp_upd = Sequential(
            Linear(2*emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU(),
            Linear(emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU()
          )

    def forward(self, h, edge_index, edge_attr):
        """
        forward 패스는 한 번의 메시지 패싱 라운드를 통해 노드 특징 `h`를 업데이트합니다.

        우리의 MPNNLayer 클래스는 PyG MessagePassing 부모 클래스를 상속하므로,
        메시지 패싱 절차를 시작하는 `propagate()` 함수를 호출하기만 하면 됩니다:
        `message()` -> `aggregate()` -> `update()`.

        MessagePassing 클래스가 구현의 대부분 로직을 처리합니다.
        커스텀 GNN을 만들기 위해, 우리는 자체적인 `message()`,
        `aggregate()`, `update()` 함수만 정의하면 됩니다(이후에 정의).

        Args:
            h: (n, d) - 초기 노드 특징
            edge_index: (e, 2) - 엣지 쌍 (i, j)
            edge_attr: (e, d_e) - 엣지 특징

        Returns:
            out: (n, d) - 업데이트된 노드 특징
        """
        out = self.propagate(edge_index, h=h, edge_attr=edge_attr)
        return out

    def message(self, h_i, h_j, edge_attr):
        """단계 (1) 메시지(Message)

        `message()` 함수는 `edge_index`의 각 엣지 (i, j)에 대해 소스 노드 j에서
        도착 노드 i로 가는 메시지를 구성합니다.

        인자들은 이해하기 약간 까다로울 수 있습니다: `message()`는
        처음에 `propagate`로 전달된 임의의 인자를 받을 수 있습니다. 또한,
        변수 이름에 `_i` 또는 `_j`를 붙여 도착 노드와 소스 노드를 구분할 수
        있는데, 예를 들어 노드 특징 `h`에 대해서는
        `h_i`와 `h_j`를 사용할 수 있습니다.

        이 부분은 `message()` 함수가 그래프의 각 엣지에 대한 메시지를
        구성하므로 이해하는 것이 매우 중요합니다. 원래 노드 특징 `h`(또는
        다른 노드 변수)의 인덱싱은 PyG가 내부적으로
        처리합니다.

        Args:
            h_i: (e, d) - 도착 노드 특징
            h_j: (e, d) - 소스 노드 특징
            edge_attr: (e, d_e) - 엣지 특징

        Returns:
            msg: (e, d) - MLP `\psi`를 통과한 메시지 `m_ij`
        """
        msg = torch.cat([h_i, h_j, edge_attr], dim=-1)
        return self.mlp_msg(msg)

    def aggregate(self, inputs, index):
        """단계 (2) 집계(Aggregate)

        `aggregate` 함수는 선택한 집계 함수(기본값은 'sum')에 따라
        이웃 노드들로부터 온 메시지를 집계합니다.

        Args:
            inputs: (e, d) - 도착 노드에서 소스 노드로 가는 메시지 `m_ij`
            index: (e, 1) - `input`의 각 엣지/메시지에 대한 소스 노드 목록

        Returns:
            aggr_out: (n, d) - 집계된 메시지 `m_i`
        """
        return scatter(inputs, index, dim=self.node_dim, reduce=self.aggr)

    def update(self, aggr_out, h):
        """
        단계 (3) 업데이트(Update)

        `update()` 함수는 집계된 메시지를 초기 노드 특징과 결합하여
        최종 노드 특징을 계산합니다.

        `update()`는 첫 번째 인자로 `aggregate()`의 결과인 `aggr_out`을
        받고, 또한 처음에 `propagate()`로 전달된 임의의 선택적 인자도
        받습니다. 예를 들어 이 경우에는 추가로 `h`를 전달합니다.

        Args:
            aggr_out: (n, d) - 집계된 메시지 `m_i`
            h: (n, d) - 초기 노드 특징

        Returns:
            upd_out: (n, d) - MLP `\phi`를 통과한 업데이트된 노드 특징
        """
        upd_out = torch.cat([h, aggr_out], dim=-1)
        return self.mlp_upd(upd_out)

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(emb_dim={self.emb_dim}, aggr={self.aggr})')

좋습니다! 우리는 앞서 소개한 방정식을 따라 **메시지 패싱 레이어**를 정의했습니다. 이 레이어를 사용하여 전체 **MPNN 그래프 속성 예측 모델**을 코딩해 봅시다. 이 모델은 분자 그래프를 입력으로 받아 여러 MPNN 레이어를 통해 처리하고, 각 그래프에 대해 단일 속성을 예측합니다.

In [ ]:
class MPNNModel(Module):
    def __init__(self, num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1):
        """그래프 속성 예측을 위한 메시지 패싱 신경망(Message Passing Neural Network) 모델

        Args:
            num_layers: (int) - 메시지 패싱 레이어 수 `L`
            emb_dim: (int) - 은닉 차원 `d`
            in_dim: (int) - 초기 노드 특징 차원 `d_n`
            edge_dim: (int) - 엣지 특징(edge feature) 차원 `d_e`
            out_dim: (int) - 출력 차원 (1로 고정)
        """
        super().__init__()

        # 초기 노드 특징에 대한 선형 투영
        # 차원: d_n -> d
        self.lin_in = Linear(in_dim, emb_dim)

        # MPNN 레이어 스택
        self.convs = torch.nn.ModuleList()
        for layer in range(num_layers):
            self.convs.append(MPNNLayer(emb_dim, edge_dim, aggr='add'))

        # 전역 풀링/리드아웃(readout) 함수 `R` (평균 풀링)
        # PyG가 `global_mean_pool()`을 통해 내부 로직을 처리
        self.pool = global_mean_pool

        # 선형 예측 헤드
        # 차원: d -> out_dim
        self.lin_pred = Linear(emb_dim, out_dim)

    def forward(self, data):
        """
        Args:
            data: (PyG.Data) - PyG 그래프 배치

        Returns:
            out: (batch_size, out_dim) - 각 그래프에 대한 예측
        """
        h = self.lin_in(data.x) # (n, d_n) -> (n, d)

        for conv in self.convs:
            h = h + conv(h, data.edge_index, data.edge_attr) # (n, d) -> (n, d)
            # 각 MPNN 레이어 뒤에 잔차 연결(residual connection)을 추가한다는 점에 유의

        h_graph = self.pool(h, data.batch) # (n, d) -> (batch_size, d)

        out = self.lin_pred(h_graph) # (batch_size, d) -> (batch_size, 1)

        return out.view(-1)

훌륭합니다! 그래프 속성 예측을 위한 첫 번째 MPNN 모델을 정의했습니다.

하지만 잠깐만요! 이 모델의 학습과 평가에 뛰어들기 전에, 모델과 레이어의 **근본적인 속성**에 대한 몇 가지 정상성 검사(sanity check)를 작성해 봅시다.

## 순열 불변성(Permutation Invariance)과 동변성(Equivariance)을 위한 단위 테스트(Unit test)

강의에서는 그래프 머신러닝의 특정 근본 속성들을 반복적으로 강조해 왔습니다:
- **GNN <ins>레이어**</ins>는 그래프 내 노드 집합의 순열에 대해 **동변(equivariant)**입니다. 즉, 노드를 순열하면 GNN이 생성하는 노드 특징도 그에 따라 순열되어야 합니다.
- 그래프 수준 속성 예측을 위한 **GNN <ins>모델**</ins>은 그래프 내 노드 집합의 순열에 대해 **불변(invariant)**입니다. 즉, 노드를 순열해도 그래프 수준의 속성은 변하지 않습니다.

(그런데 잠깐...**순열(permutation)이란 무엇일까요?** 본질적으로 그것은 그래프 내 **노드의 순서 배열(ordering of the nodes)**입니다. 텍스트나 이미지 데이터와 달리, 일반적으로 노드에 순서를 할당하는 **정준적(canonical) 방법은 없습니다**. 그러나 그래프에 대해 머신러닝을 수행하려면(이것이 바로 이 과목의 주제입니다!) 그래프를 컴퓨터에 저장하고 처리해야 합니다. 따라서 우리 모델이 이러한 **정준적 순서의 부재**, 즉 그래프 노드의 순열을 원칙적으로 다룰 수 있도록 보장해야 합니다. 이것이 위 진술들이 말하고자 하는 바입니다.)

### 형식화(Formalism)

순열 불변성과 동변성에 대한 이러한 개념을 행렬 표기법으로 형식화해 봅시다(그렇게 하는 것이 더 쉽습니다).

- $\mathbf{H} \in \mathbb{R}^{n \times d}$를 주어진 분자 그래프에 대한 노드 특징 행렬이라고 하자. 여기서 $n$은 노드/원자의 수이고 각 행 $h_i$는 노드 $i$에 대한 $d$차원 특징입니다.
- $\mathbf{A} \in \mathbb{R}^{n \times n}$를 인접 행렬(adjacency matrix)이라고 하자. 각 항목 $a_{ij}$는 노드 $i$와 $j$ 사이의 엣지 존재 여부를 나타냅니다.
- $\mathbf{F}(\mathbf{H}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}^{n \times d}$를 노드 특징과 인접 행렬을 입력으로 받아 **업데이트된 노드 특징**을 반환하는 **GNN <ins>레이어**</ins>라고 하자.
- $f(\mathbf{H}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}$를 노드 특징과 인접 행렬을 입력으로 받아 **예측된 그래프 수준 속성**을 반환하는 **GNN <ins>모델**</ins>이라고 하자.
- $\mathbf{P} \in \mathbb{R}^{n \times n}$를 모든 행과 열에 정확히 하나의 1이 있고 나머지는 0인 **[순열 행렬(permutation matrix)](https://en.wikipedia.org/wiki/Permutation_matrix)**이라고 하자. $\mathbf{P}$를 행렬에 왼쪽 곱하면 행렬의 행 순서가 바뀝니다.

### 순열 동변성(Permutation Equivariance)

GNN <ins>레이어</ins> $\mathbf{F}$는 다음과 같이 **순열 동변(permutation equivariant)**합니다:
$$
\mathbf{F}(\mathbf{PH}, \mathbf{PAP^T}) = \mathbf{P} \ \mathbf{F}(\mathbf{H}, \mathbf{A}).
$$

위 식을 다르게 표현하는 방법은 다음과 같습니다: (1) 업데이트된 노드 특징 $\mathbf{H'} = \mathbf{F}(\mathbf{H}, \mathbf{A})$를 고려하자. (2) GNN 레이어 $\mathbf{F}$의 입력에 임의의 순열 행렬 $\mathbf{P}$를 적용하는 것은 $\mathbf{H'}$에 동일한 순열을 적용하는 것과 같은 결과를 내야 합니다:
$$
\mathbf{F}(\mathbf{PH}, \mathbf{PAP^T}) = \mathbf{P} \ \mathbf{H'}
$$

### 순열 불변성(Permutation Invariance)

그래프 수준 예측을 위한 GNN <ins>모델</ins> $f$는 다음과 같이 **순열 불변(permutation invariant)**합니다:
$$
f(\mathbf{PH}, \mathbf{PAP^T}) = f(\mathbf{H}, \mathbf{A}).
$$

위 식을 다르게 표현하는 방법은 다음과 같습니다: (1) 예측된 분자 속성 $\mathbf{\hat y} = f(\mathbf{H}, \mathbf{A})$를 고려하자. (2) GNN 모델 $f$의 입력에 임의의 순열 행렬 $\mathbf{P}$를 적용하는 것은 적용하지 않은 것과 같은 결과를 내야 합니다:
$$
f(\mathbf{PH}, \mathbf{PAP^T}) = \mathbf{\hat y}.
$$

이러한 형식화를 마쳤으니, 우리의 `MPNNModel`과 `MPNNLayer`가 실제로 각각 순열 불변성과 동변성을 갖는지 확인하기 위한 몇 가지 단위 테스트를 작성해 봅시다.

In [ ]:
def permute_graph(data, perm):
    """PyG Data 객체의 속성들을 일관되게 순열하기 위한 헬퍼 함수.
    """
    # 노드 속성 순서를 순열
    data.x = data.x[perm]
    data.pos = data.pos[perm]
    data.z = data.z[perm]
    data.batch = data.batch[perm]

    # 엣지 인덱스를 순열
    adj = to_dense_adj(data.edge_index)
    adj = adj[:, perm, :]
    adj = adj[:, :, perm]
    data.edge_index = dense_to_sparse(adj)[0]

    # 참고:
    # (1) 우리는 원래 순열 행렬 P를 0과 1 항목만 갖는 것으로 정의했지만,
    #     `perm`을 통한 구현은 대신 torch 텐서에 대한 인덱싱을 사용합니다.
    # (2) edge_attr를 순열하는 것은 번거로우므로, 이를 상수
    #     더미 값으로 설정합니다. 단위 테스트를 넘어선 모든 실험에서는, 모든 GNN 모델이
    #     원래의 edge_attr를 사용합니다.

    return data

def permutation_invariance_unit_test(module, dataloader):
    """모듈(GNN 모델)이 순열 불변인지 확인하기 위한
    단위 테스트.
    """
    it = iter(dataloader)
    data = next(it)

    # edge_attr를 더미 값으로 설정(단순화를 위해)
    data.edge_attr = torch.zeros(data.edge_attr.shape)

    # 원래 예제에 대한 forward 패스
    out_1 = module(data)

    # 무작위 순열 생성
    perm = torch.randperm(data.x.shape[0])
    data = permute_graph(data, perm)

    # 순열된 예제에 대한 forward 패스
    out_2 = module(data)

    # 변환 적용 후 출력이 달라지는지 확인
    return torch.allclose(out_1, out_2, atol=1e-04)


def permutation_equivariance_unit_test(module, dataloader):
    """모듈(GNN 레이어)이 순열 동변인지 확인하기 위한
    단위 테스트.
    """
    it = iter(dataloader)
    data = next(it)

    # edge_attr를 더미 값으로 설정(단순화를 위해)
    data.edge_attr = torch.zeros(data.edge_attr.shape)

    # 원래 예제에 대한 forward 패스
    out_1 = module(data.x, data.edge_index, data.edge_attr)

    # 무작위 순열 생성
    perm = torch.randperm(data.x.shape[0])
    data = permute_graph(data, perm)

    # 순열된 예제에 대한 forward 패스
    out_2 = module(data.x, data.edge_index, data.edge_attr)

    # 변환 적용 후 출력이 달라지는지 확인
    return torch.allclose(out_1[perm], out_2, atol=1e-04)

이제 (전체 MPNN 모델에 대한) 순열 불변성과 (MPNN 레이어에 대한) 순열 동변성을 위한 단위 테스트를 정의했으니, 정상성 검사(sanity check)를 수행해 봅시다:

In [ ]:
# 단위 테스트를 위한 임시 모델, 레이어, 데이터로더를 인스턴스화
layer = MPNNLayer(emb_dim=11, edge_dim=4)
model = MPNNModel(num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1)
dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# MPNN 모델에 대한 순열 불변성 단위 테스트
print(f"Is {type(model).__name__} permutation invariant? --> {permutation_invariance_unit_test(model, dataloader)}!")

# MPNN 레이어에 대한 순열 동변성 단위 테스트
print(f"Is {type(layer).__name__} permutation equivariant? --> {permutation_equivariance_unit_test(layer, dataloader)}!")

## 모델 학습 및 평가

좋습니다! 마침내 QM9에서 모델을 학습하고 평가할 준비가 되었습니다. 우리는 모델과 데이터로더를 입력으로 받아 학습을 수행하고 **검증 세트(validation set)**와 **테스트 세트(test set)**에 대한 최종 성능을 반환하는 **기본 실험 루프**를 제공했습니다.

우리는 은닉 차원이 64인 4개의 메시지 패싱 레이어로 구성된 `MPNNModel`을 학습할 것입니다.

In [ ]:
#@title [RUN] Helper functions for managing experiments, training, and evaluating models.

def train(model, train_loader, optimizer, device):
    model.train()
    loss_all = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        y_pred = model(data)
        loss = F.mse_loss(y_pred, data.y)
        loss.backward()
        loss_all += loss.item() * data.num_graphs
        optimizer.step()
    return loss_all / len(train_loader.dataset)


def eval(model, loader, device):
    model.eval()
    error = 0

    for data in loader:
        data = data.to(device)
        with torch.no_grad():
            y_pred = model(data)
            # std를 사용한 평균 절대 오차(Mean Absolute Error) (데이터 준비 시 계산됨)
            error += (y_pred * std - data.y * std).abs().sum().item()
    return error / len(loader.dataset)


def run_experiment(model, model_name, train_loader, val_loader, test_loader, n_epochs=100):

    print(f"Running experiment for {model_name}, training on {len(train_loader.dataset)} samples for {n_epochs} epochs.")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("\nModel architecture:")
    print(model)
    total_param = 0
    for param in model.parameters():
        total_param += np.prod(list(param.data.size()))
    print(f'Total parameters: {total_param}')
    model = model.to(device)

    # 학습률(LR) 1e-3의 Adam 옵티마이저
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # 검증 지표가 개선되지 않을 때 학습률을 감소시키는 LR 스케줄러
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.9, patience=5, min_lr=0.00001)

    print("\nStart training:")
    best_val_error = None
    perf_per_epoch = [] # 에폭별 Test/Val MAE 추적 (플로팅용)
    t = time.time()
    for epoch in range(1, n_epochs+1):
        # 각 에폭 시작 시 LR 스케줄러 호출
        lr = scheduler.optimizer.param_groups[0]['lr']

        # 한 에폭 동안 모델 학습, 평균 학습 손실 반환
        loss = train(model, train_loader, optimizer, device)

        # 검증 세트에서 모델 평가
        val_error = eval(model, val_loader, device)

        if best_val_error is None or val_error <= best_val_error:
            # 검증 지표가 개선되면 테스트 세트에서 모델 평가
            test_error = eval(model, test_loader, device)
            best_val_error = val_error

        if epoch % 10 == 0:
            # 10 에폭마다 통계를 출력하고 추적
            print(f'Epoch: {epoch:03d}, LR: {lr:5f}, Loss: {loss:.7f}, '
                  f'Val MAE: {val_error:.7f}, Test MAE: {test_error:.7f}')

        scheduler.step(val_error)
        perf_per_epoch.append((test_error, val_error, epoch, model_name))

    t = time.time() - t
    train_time = t/60
    print(f"\nDone! Training took {train_time:.2f} mins. Best validation MAE: {best_val_error:.7f}, corresponding test MAE: {test_error:.7f}.")

    return best_val_error, test_error, train_time, perf_per_epoch

In [ ]:
model = MPNNModel(num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1)
model_name = type(model).__name__
best_val_error, test_error, train_time, perf_per_epoch = run_experiment(
    model,
    model_name,
    train_loader,
    val_loader,
    test_loader,
    n_epochs=100
)
RESULTS[model_name] = (best_val_error, test_error, train_time)
df_temp = pd.DataFrame(perf_per_epoch, columns=["Test MAE", "Val MAE", "Epoch", "Model"])
DF_RESULTS = DF_RESULTS.append(df_temp, ignore_index=True)

In [ ]:
RESULTS

In [ ]:
p = sns.lineplot(x="Epoch", y="Val MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 2));

In [ ]:
p = sns.lineplot(x="Epoch", y="Test MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 1));

훌륭합니다! 지금까지의 모든 내용은 이미 강의에서 다루었으며, 함께 제공된 코드와 더불어 이번 실습이 지금까지 유용한 복습이 되었기를 바랍니다.

이제 지금까지 공부한 내용을 스스로 생각해 봐야 하는 재미있는 부분입니다!

---
---
---

# 🧊 Part 1: 기하 그래프(Geometric Graphs)와 3D 좌표를 활용한 메시지 패싱

분자 그래프의 각 원자에 **3D 좌표**가 주어졌다는 것을 기억하시나요?

분자 그래프와, 자연에서 나타나는 다른 구조화된 데이터는 단순히 평평한 평면 위에 존재하지 않습니다. 대신, 분자는 그 속성과 기능에 영향을 미치는 **고유한 3D 구조**를 갖습니다.

QM9에서 분자 하나를 3D의 모든 영광 속에서 시각화해 봅시다!

마우스 커서로 이 분자를 움직여 보세요!

In [ ]:
MolTo3DView(smi2conf(Chem.MolToSmiles(to_rdkit(train_dataset[48]))))

## 💻**Task 1.1:** 원자 좌표를 노드 특징으로 통합하는 메시지 패싱 신경망을 개발하세요 **(0.5 Marks)**.


우리의 초기이자 다소 **'바닐라(vanilla)' MPNN**인 `MPNNModel`은 원자 좌표를 무시하고 노드 특징만 사용하여 메시지 패싱을 수행합니다. 이는 모델이 목표 속성을 예측하는 데 유용한 **3D 구조 정보**를 활용하고 있지 **않다**는 것을 의미합니다.

여러분의 첫 번째 과제는 원래의 `MPNNModel`을 수정하여 **원자 좌표**를 **노드 특징**에 통합하는 것입니다.

새로운 `CoordMPNNModel` 클래스의 대부분을 정의해 두었으며, 여러분은 `YOUR CODE HERE` 섹션을 채워야 합니다.

🤔 *힌트: 다시 상기하자면, 3D 원자 위치는 `data.pos`에 저장되어 있습니다. 지금 당장 매우 영리한 무언가를 할 필요는 없습니다(그것은 나중에 다룹니다). **단순한** 해법으로 시작해도 괜찮습니다. 예를 들어 연결(concatenation)이나 합(summation) 등이 있습니다.*


In [ ]:
class CoordMPNNModel(MPNNModel):
    def __init__(self, num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1):
        """그래프 속성 예측을 위한 메시지 전달 신경망(Message Passing Neural Network) 모델

        이 모델은 노드 특징(node features)과 좌표(coordinates)를 모두 입력으로 사용합니다.

        Args:
            num_layers: (int) - 메시지 전달 층의 개수 `L`
            emb_dim: (int) - 은닉 차원 `d`
            in_dim: (int) - 초기 노드 특징 차원 `d_n`
            edge_dim: (int) - 엣지 특징 차원 `d_e`
            out_dim: (int) - 출력 차원 (1로 고정)
        """
        super().__init__()

        # ============ 여기에 코드를 작성하세요 ==============
        # 원자 위치(atom positions)를 반영하도록 입력 선형 층을 수정하거나
        # 새로운 입력 층을 추가하세요.
        #
        # 초기 노드 특징과 좌표를 위한 선형 사영(linear projection)
        # 차원: ??? -> d
        # self.lin_in = ...
        # ==========================================

        # MPNN 층의 스택
        self.convs = torch.nn.ModuleList()
        for layer in range(num_layers):
            self.convs.append(MPNNLayer(emb_dim, edge_dim, aggr='add'))

        # 전역 풀링/리드아웃 함수 `R` (평균 풀링)
        # PyG가 `global_mean_pool()`을 통해 내부 로직을 처리합니다
        self.pool = global_mean_pool

        # 선형 예측 헤드(prediction head)
        # 차원: d -> out_dim
        self.lin_pred = Linear(emb_dim, out_dim)

    def forward(self, data):
        """
        Args:
            data: (PyG.Data) - PyG 그래프의 배치

        Returns:
            out: (batch_size, out_dim) - 각 그래프에 대한 예측값
        """
        # ============ 여기에 코드를 작성하세요 ==============
        # 원자 위치를 특징과 함께 통합하세요.
        #
        # h = ...
        # ==========================================

        for conv in self.convs:
            h = h + conv(h, data.edge_index, data.edge_attr) # (n, d) -> (n, d)
            # 각 MPNN 층 뒤에 잔차 연결(residual connection)을 추가한다는 점에 유의하세요

        h_graph = self.pool(h, data.batch) # (n, d) -> (batch_size, d)

        out = self.lin_pred(h_graph) # (batch_size, d) -> (batch_size, 1)

        return out.view(-1)

## 💻**Task 1.2:** 노드 특징과 좌표를 갖는 새로운 `CoordMPNNModel`과, 그것을 구성하는 `MPNNLayer`의 순열 불변성 및 동변성 속성을 테스트하세요. **(0.5 Marks)**

훌륭합니다! 여러분은 분자 속성을 예측하기 위해 **원자 특징**과 **좌표**를 모두 활용하는 MPNN을 성공적으로 구현했습니다.

이를 평가하기 전에, 모델과 레이어가 모든 기본 GNN을 구성하는 바람직한 속성들을 갖는지 확인하기 위해 다시 한번 순열 정상성 검사를 실행해 봅시다:
- `MPNNLayer`는 순열 동변이어야 합니다(앞서 이미 보였지만, **철저히** 이해하기 위해 이 연습을 반복하기를 바랍니다).
- `CoordMPNNModel`은 순열 불변이어야 합니다.

여러분의 과제는 필요한 단위 테스트를 실행하기 위해 `YOUR CODE HERE` 섹션을 채우는 것입니다. 아직 새로운 단위 테스트를 작성할 필요는 없으며, 앞서 정의한 것들을 재사용할 수 있습니다.

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 단위 테스트를 위한 임시 모델, 층, 데이터로더를 인스턴스화하세요.
# 이제 우리는 CoordMPNNModel을 단위 테스트하고 있으며, 이 모델은 이전 모델과는
# 다르지만 여전히 MPNNLayer로 구성되어 있다는 점을 기억하세요.
#
# layer = ...
# model = ...
# ==========================================
dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# MPNN 모델에 대한 순열 불변성(permutation invariance) 단위 테스트
print(f"Is {type(model).__name__} permutation invariant? --> {permutation_invariance_unit_test(model, dataloader)}!")

# MPNN 층에 대한 순열 등변성(permutation equivariance) 단위 테스트
print(f"Is {type(layer).__name__} permutation equivariant? --> {permutation_equivariance_unit_test(layer, dataloader)}!")

## 💻**Task 1.3.** 새로운 `CoordMPNNModel`이 노드 특징과 노드 좌표의 순열(permutation) 모두에 대해 불변(invariant)임을 증명하세요. **(0.5 Marks)**

🤔 *힌트: 우리는 기본(vanilla) MPNN 모델에 대해 순열 불변성을 형식화했던 방식을 따르는 간단한 진술을 기대합니다. 여러분은 대부분의 형식화를 복사해 붙여넣되, 여러분의 MPNN이 노드 특징과 좌표를 모두 어떻게 통합하는지를 반영하면 됩니다. 추가로 주어진 분자 그래프에 대한 노드 좌표 행렬로서 $\mathbf{X} \in \mathbb{R}^{n \times 3}$를 도입할 수 있습니다.*


---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

## 💻**Task 1.4.** 노드 특징과 좌표를 갖는 여러분의 `CoordMPNNModel`을 QM9에서 학습하고 평가하세요. **(0.5 Marks)**

훌륭합니다! 이제 노드 특징과 좌표를 갖는 새로운 MPNN을 QM9에서 학습하고 평가할 준비가 되었습니다.

우리가 제공한 실험 루프를 재사용하고 `YOUR CODE HERE` 섹션을 채워 실험을 실행하세요.

이전의 바닐라 `MPNNModel`과 결과를 공정하게 비교하기 위해, 은닉 차원이 64인 4개의 메시지 패싱 레이어로 구성된 `CoordMPNNModel`을 학습할 것입니다.

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 적절한 인자를 사용하여 CoordMPNNModel을 인스턴스화하세요.
#
# model = CoordMPNNModel(...)
# ==========================================

model_name = type(model).__name__
best_val_error, test_error, train_time, perf_per_epoch = run_experiment(
    model,
    model_name, # "MPNN w/ Features and Coordinates",
    train_loader,
    val_loader,
    test_loader,
    n_epochs=100
)

RESULTS[model_name] = (best_val_error, test_error, train_time)
df_temp = pd.DataFrame(perf_per_epoch, columns=["Test MAE", "Val MAE", "Epoch", "Model"])
DF_RESULTS = DF_RESULTS.append(df_temp, ignore_index=True)

In [ ]:
RESULTS

In [ ]:
p = sns.lineplot(x="Epoch", y="Val MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 2));

In [ ]:
p = sns.lineplot(x="Epoch", y="Test MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 1));

음... 지금까지 `CoordMPNNModel`을 올바르게 구현했다면, 매우 흥미로운 결과를 보게 될 것입니다 -- `CoordMPNNModel`의 성능이 바닐라 `MPNNModel`과 거의 비슷하거나 약간 더 나쁩니다!

<font color='red'>이는 `CoordMPNNModel`이 3D 구조 정보를 원칙적인 방식으로 사용하고 있지 않기 때문입니다.</font>

다음 섹션들은 이런 일이 왜 발생하는지 형식화하고 이해하는 데 도움을 줄 것입니다.

---
---
---

# 🔄 Part 2: 3D 대칭에 대한 불변성: 회전과 평행이동

우리는 `CoordMPNNModel`이 노드 특징과 좌표를 모두 사용함에도 불구하고 `MPNNModel`에 비해 성능이 예상외로 평범하다는 것을 보았습니다. (그러나 여러분의 결과가 다르게 나오더라도 당황하지 마세요.) 그 이유를 알아내기 위해, 우리는 **3D 대칭(3D symmetries)**의 개념을 이해해야 합니다.

### 기하 불변성(Geometric Invariance)

분자 그래프는 각 원자에 대해 3D 좌표를 갖는다는 것을 떠올려 봅시다. 지금까지 의도적으로 여러분에게 숨겨 온(😈) 핵심적인 세부 사항은, 이 3D 좌표가 **본질적으로 고정되어 있거나** **영구적이지 않다**는 것입니다. 대신, 이들은 **기준 좌표계(frame of reference)**를 기준으로 **실험적으로 결정**되었습니다.

이 진술들을 완전히 파악하기 위해, 약물 유사 분자들이 3D 공간에서 움직이는 GIF가 여기 있습니다...

<!-- ![](https://drive.google.com/uc?id=1QcQcF91TD-CTKFaR4NN8YyXbtSTcGny8) -->
<img src="https://github.com/chaitjo/dump/raw/main/3d-molecule-moving.gif">

원자들의 3D 좌표는 끊임없이 **회전**하고 **평행이동**하고 있습니다. 그러나 이 분자의 **속성**은 우리가 그것을 어떻게 회전하거나 평행이동하든 항상 동일하게 유지됩니다. 다시 말해, 분자의 속성은 3D 회전과 평행이동에 대해 **불변(invariant)**합니다.

이 블록에서 우리는 이러한 규칙성을 존중하는 GNN 레이어와 모델을 설계하는 방법을 공부할 것입니다.

### 형식화(Formalism)

GNN에서 3D 회전과 평행이동에 대한 불변성 개념을 행렬 표기법으로 형식화해 봅시다.

- $\mathbf{H} \in \mathbb{R}^{n \times d}$를 주어진 분자 그래프에 대한 노드 특징 행렬이라고 하자. 여기서 $n$은 노드/원자의 수이고 각 행 $h_i$는 노드 $i$에 대한 $d$차원 특징입니다.
- $\mathbf{X} \in \mathbb{R}^{n \times 3}$를 주어진 분자 그래프에 대한 노드 좌표 행렬이라고 하자. 여기서 $n$은 노드/원자의 수이고 각 행 $x_i$는 노드 $i$에 대한 3D 좌표입니다.
- $\mathbf{A} \in \mathbb{R}^{n \times n}$를 인접 행렬(adjacency matrix)이라고 하자. 각 항목 $a_{ij}$는 노드 $i$와 $j$ 사이의 엣지 존재 여부를 나타냅니다.
- $\mathbf{F}(\mathbf{H}, \mathbf{X}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times 3} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}^{n \times d}$를 노드 특징, 노드 좌표, 인접 행렬을 입력으로 받아 **업데이트된 노드 특징**을 반환하는 **GNN <ins>레이어**</ins>라고 하자.
- $f(\mathbf{H}, \mathbf{X}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times 3} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}$를 노드 특징, 노드 좌표, 인접 행렬을 입력으로 받아 **예측된 그래프 수준 속성**을 반환하는 **GNN <ins>모델**</ins>이라고 하자.

(GNN 레이어 $\mathbf{F}$와 GNN 모델 $\mathbf{f}$의 표기법을 노드 좌표 행렬 $\mathbf{X}$를 추가 입력으로 포함하도록 업데이트했음에 유의하세요.)

## 💻**Task 2.1:** GNN <ins>모델</ins> $f$와 GNN <ins>층(layer)</ins> $\mathbf{F}$가 3D 회전(rotations)과 평행이동(translations)에 대해 불변(invariant)이라는 것은 무엇을 의미하나요? 위의 정의들을 사용하여 이를 _수학적으로_ 표현하세요. **(0.5 Mark)**

🤔 *힌트: 순열 불변성과 등변성에 대한 형식화를 다시 살펴보면 이를 어떻게 접근할지 감을 잡을 수 있습니다. 위에서 제공한 행렬 표기법을 사용해야 합니다. 순열 행렬 $\mathbf{P}$와 유사하게, 이제 답에서 직교 [**회전 행렬(rotation matrix)**](https://en.wikipedia.org/wiki/Rotation_matrix) $\mathbf{Q} \in \mathbb{R}^{3 \times 3}$와 [**평행이동 벡터(translation vector)**](https://en.wikipedia.org/wiki/Translation_(geometry)) $\mathbf{t} \in \mathbb{R}^3$를 정의할 수 있습니다. 이들은 노드 좌표 행렬 $\mathbf{X} \in \mathbb{R}^{n \times 3}$에 작용하게 됩니다*.

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

보다 원칙적인(principled) MPNN 모델을 코딩하기 시작하기 전에, 분자 속성을 예측하는 GNN에서 3D 회전과 평행이동에 대한 불변성이 왜 바람직한 것인지 잠시 생각해보길 바랍니다...

## 💻**Task 2.2:** 3D 회전과 평행이동에 대한 불변성은 GNN에 바람직한 속성인가요? 그 이유를 설명하세요. **(0.5 Marks)**

🤔 *힌트: 우리는 에세이를 원하는 것이 아니며, 여기서는 몇 문장이면 충분합니다.*

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

## 💻**Task 2.3:** `CoordMPNNModel`이 3D 회전 및 평행이동에 대해 불변인지 확인하는 단위 테스트(unit test)를 작성하세요. **(0.5 Mark)**


🤔 *힌트: 다음의 경우에 모델의 출력이 변하는지 보이세요:*
1. `data.pos`의 모든 원자 좌표에 임의의 _직교_ 회전 행렬 $Q \in \mathbb{R}^{3 \times 3}$를 곱한 경우. (회전 행렬을 만드는 헬퍼 함수를 제공했습니다.)
2. `data.pos`의 모든 원자 좌표를 임의의 평행이동 벡터 $\mathbf{t} \in \mathbb{R}^3$만큼 이동시킨 경우.

In [ ]:
def random_orthogonal_matrix(dim=3):
  """(dim, dim) 형태의 무작위 직교 행렬을 생성하는 헬퍼 함수
  """
  Q = torch.tensor(ortho_group.rvs(dim=dim)).float()
  return Q


def rot_trans_invariance_unit_test(module, dataloader):
    """모듈(GNN 모델/레이어)이 회전 및 평행이동에
    대해 불변(invariant)인지 확인하기 위한 단위 테스트.
    """
    it = iter(dataloader)
    data = next(it)

    # 원본 예제에 대한 순전파
    # 참고: 동일한 단위 테스트를 GNN 모델과 레이어 모두에
    #       사용할 수 있도록 조건부 순전파를 작성했습니다.
    #       레이어용 기능은 이후에 유용하게 쓰입니다.
    if isinstance(module, MPNNModel):
        out_1 = module(data)
    else: # if ininstance(module, MessagePassing):
        out_1 = module(data.x, data.pos, data.edge_index, data.edge_attr)

    Q = random_orthogonal_matrix(dim=3)
    t = torch.rand(3)
    # ============ 여기에 코드를 작성하세요 ==============
    # data에 무작위 회전 + 평행이동을 수행하세요.
    #
    # data.pos = ...
    # ==========================================

    # 회전 + 평행이동된 예제에 대한 순전파
    if isinstance(module, MPNNModel):
        out_2 = module(data)
    else: # if ininstance(module, MessagePassing):
        out_2 = module(data.x, data.pos, data.edge_index, data.edge_attr)

    # ============ 여기에 코드를 작성하세요 ==============
    # 변환을 적용한 후 출력이 변하는지 확인하세요.
    #
    # return torch.allclose(..., atol=1e-04)
    # ==========================================

이제 회전 및 평행이동 불변성에 대한 단위 테스트를 정의했으니, `CoordMPNNModel`에 대해 정상성 검사(sanity check)를 수행하세요:

(스포일러 주의: 예상대로 구현했다면, 단위 테스트는 `CoordMPNNModel`에 대해 `False`를 반환해야 합니다.)

In [ ]:
# 단위 테스트를 위한 임시 모델, 레이어, 데이터로더 인스턴스화
model = CoordMPNNModel(num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1)
dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# MPNN 모델에 대한 회전 및 평행이동 불변성 단위 테스트
print(f"Is {type(model).__name__} rotation and translation invariant? --> {rot_trans_invariance_unit_test(model, dataloader)}!")

이 파트에서는 GNN이 3D 회전 및 평행이동에 어떻게 불변일 수 있는지 형식화하고, 이것이 분자 특성 예측에 왜 바람직한지 생각해 보았으며, `CoordMPNNModel`이 회전 및 평행이동에 불변이 아니었음을 보였습니다.

이 시점에서, 여러분은 `CoordMPNNModel`의 성능이 기본 `MPNNModel`과 같거나 더 나쁜 이유, 그리고 이 파트를 시작하기 전에 했던 처음의 진술이 의미하는 바를 구체적으로 이해하게 되었을 것입니다:
>"`CoordMPNNModel`은 3D 구조 정보를 원칙적인 방식으로 사용하고 있지 않다"

다음 파트에서 이를 고쳐 봅시다!

---
---
---

# ✈️ Part 3: 3D 회전 및 평행이동에 대한 불변성을 갖는 메시지 패싱(Message Passing)

이 섹션에서는 3D 좌표를 가진 그래프에 대해 작동하는 GNN 모델을 이론적으로 더 타당한 방식으로 설계하는 방법을 깊이 살펴봅니다.

## 💻**Task 3.1:** 3D 회전 및 평행이동에 모두 <ins>불변(invariant)</ins>인 새로운 메시지 패싱 레이어와 그에 따른 MPNN 모델을 설계하세요. **(2 Marks)**

**❗️ 참고:** 이 문제에는 단 하나의 정답이 있는 것은 아닙니다.

우리의 초기 **'바닐라(vanilla)' MPNN**인 `MPNNModel`과 `MPNNLayer`는 원자 좌표를 무시하고 노드 특징(node features)만을 사용하여 메시지 패싱을 수행했습니다. 이는 모델이 타깃 특성을 예측하는 데 **3D 구조 정보**를 활용하고 있지 **않았음**을 의미합니다.

우리의 두 번째 **'순진한(naive)' 좌표 MPNN**인 `CoordMPNNModel`은 노드 특징과 원자 좌표를 원칙 없는 방식으로 함께 사용했고, 그 결과 모델이 좌표의 3D 회전 및 평행이동에 불변이 아니게 되었습니다(이전 파트에서 보았듯이, 불변성은 바람직한 성질이었습니다).

여러분의 과제는 **원자 좌표**와 **노드 특징**을 모두 활용하는 새로운 `InvariantMPNNLayer`를 정의하는 것입니다.

새로운 `InvariantMPNNLayer`의 대부분은 우리가 정의해 두었으며, 여러분은 `YOUR CODE HERE` 섹션을 채워야 합니다. 또한 여러분의 새 레이어를 인스턴스화하여 모델을 구성하는 `InvariantMPNNModel`도 이미 정의해 두었습니다. 여러분은 새 레이어만 정의하면 됩니다.

🤔 *힌트 1: 이전의 `CoordMPNNModel`과 달리, 좌표 정보를 노드 특징에 통합하는 대신 좌표 정보를 사용하여 메시지를 구성하기를 권합니다. 특히, 메시지를 구성하기 위해 좌표를 원칙적인 방식으로 **어떻게** 사용할지 생각해 보기를 바랍니다: 좌표 쌍을 사용하여 계산할 수 있으면서 그것들을 회전하고 평행이동해도 불변인 측정량은 무엇일까요?*

🤔 *힌트 2: `propagate()`로 전달되는 텐서는 변수 이름에 `_i` 또는 `_j`를 붙여 각각의 노드에 매핑할 수 있습니다. 예를 들어 노드 특징 `h`에 대해 `h_i`와 `h_j`처럼 말입니다. 일반적으로 `_i`는 정보를 집계하는 중심 노드를 가리키고, `_j`는 이웃 노드를 가리킨다는 점에 유의하세요.*

In [ ]:
class InvariantMPNNLayer(MessagePassing):
    def __init__(self, emb_dim=64, edge_dim=4, aggr='add'):
        """메시지 패싱 신경망 레이어 (Message Passing Neural Network Layer)

        이 레이어는 3D 회전 및 평행이동에 대해 불변(invariant)입니다.

        Args:
            emb_dim: (int) - 은닉 차원 `d`
            edge_dim: (int) - 엣지 특징 차원 `d_e`
            aggr: (str) - 집계 함수 `\oplus` (sum/mean/max)
        """
        # 집계 함수 설정
        super().__init__(aggr=aggr)

        self.emb_dim = emb_dim
        self.edge_dim = edge_dim

        # ============ 여기에 코드를 작성하세요 ==============
        # 메시지 `m_ij`를 계산하기 위한 MLP `\psi`
        # 차원: (???) -> d
        #
        # self.mlp_msg = Sequential(...)
        # ==========================================

        # 갱신된 노드 특징 `h_i^{l+1}`를 계산하기 위한 MLP `\phi`
        # 차원: 2d -> d
        self.mlp_upd = Sequential(
            Linear(2*emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU(),
            Linear(emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU()
          )

    def forward(self, h, pos, edge_index, edge_attr):
        """
        순전파는 한 번의 메시지 패싱 라운드를 통해 노드 특징 `h`를 갱신합니다.

        Args:
            h: (n, d) - 초기 노드 특징
            pos: (n, 3) - 초기 노드 좌표
            edge_index: (e, 2) - 엣지 쌍 (i, j)
            edge_attr: (e, d_e) - 엣지 특징

        Returns:
            out: (n, d) - 갱신된 노드 특징
        """
        # ============ 여기에 코드를 작성하세요 ==============
        # `forward()` 함수에 초기 노드 좌표를 나타내는 새로운 인자
        # `pos`가 추가된 점에 주목하세요. 여러분의 과제는 `pos`를
        # 다른 인자들과 함께 `message()` 함수로 전달하도록
        # `propagate()` 함수를 갱신하는 것입니다.
        #
        # out = self.propagate(...)
        # return out
        # ==========================================

    # ============ 여기에 코드를 작성하세요 ==============
    # 소스 노드 및 목적지 노드 특징, 노드 좌표, `edge_attr`를 인자로
    # 받는 사용자 정의 `message()` 함수를 작성하세요.
    # 메시지가 회전 및 평행이동에 대해 불변이 되도록 좌표 `pos`를
    # 메시지 계산에 통합하세요.
    # 이렇게 하면 전체 레이어도 불변이 됩니다.
    #
    # def message(self, ...):
    # """`message()` 함수는 `edge_index`의 각 엣지 (i, j)에 대해
    #    소스 노드 j에서 목적지 노드 i로 가는 메시지를 구성합니다.
    #
    #    Args:
    #        ...
    #
    #    Returns:
    #        ...
    # """
    #   ...
    #   msg = ...
    #   return self.mlp_msg(msg)
    # ==========================================

    def aggregate(self, inputs, index):
        """`aggregate` 함수는 선택된 집계 함수(기본값은 'sum')에 따라
        이웃 노드들로부터의 메시지를 집계합니다.

        Args:
            inputs: (e, d) - 목적지 노드에서 소스 노드로의 메시지 `m_ij`
            index: (e, 1) - `input`의 각 엣지/메시지에 대한 소스 노드 목록

        Returns:
            aggr_out: (n, d) - 집계된 메시지 `m_i`
        """
        return scatter(inputs, index, dim=self.node_dim, reduce=self.aggr)

    def update(self, aggr_out, h):
        """`update()` 함수는 집계된 메시지를 초기 노드 특징과 결합하여
        최종 노드 특징을 계산합니다.

        Args:
            aggr_out: (n, d) - 집계된 메시지 `m_i`
            h: (n, d) - 초기 노드 특징

        Returns:
            upd_out: (n, d) - MLP `\phi`를 통과한 갱신된 노드 특징
        """
        upd_out = torch.cat([h, aggr_out], dim=-1)
        return self.mlp_upd(upd_out)

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(emb_dim={self.emb_dim}, aggr={self.aggr})')


class InvariantMPNNModel(MPNNModel):
    def __init__(self, num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1):
        """그래프 속성 예측을 위한 메시지 패싱 신경망 모델

        이 모델은 노드 특징과 좌표를 모두 입력으로 사용하며,
        3D 회전 및 평행이동에 대해 불변(invariant)입니다.

        Args:
            num_layers: (int) - 메시지 패싱 레이어 수 `L`
            emb_dim: (int) - 은닉 차원 `d`
            in_dim: (int) - 초기 노드 특징 차원 `d_n`
            edge_dim: (int) - 엣지 특징 차원 `d_e`
            out_dim: (int) - 출력 차원 (1로 고정)
        """
        super().__init__()

        # 초기 노드 특징에 대한 선형 사영
        # 차원: d_n -> d
        self.lin_in = Linear(in_dim, emb_dim)

        # 불변 MPNN 레이어 스택
        self.convs = torch.nn.ModuleList()
        for layer in range(num_layers):
            self.convs.append(InvariantMPNNLayer(emb_dim, edge_dim, aggr='add'))

        # 전역 풀링/리드아웃 함수 `R` (평균 풀링)
        # PyG는 `global_mean_pool()`을 통해 내부 로직을 처리합니다
        self.pool = global_mean_pool

        # 선형 예측 헤드
        # 차원: d -> out_dim
        self.lin_pred = Linear(emb_dim, out_dim)

    def forward(self, data):
        """
        Args:
            data: (PyG.Data) - PyG 그래프 배치

        Returns:
            out: (batch_size, out_dim) - 각 그래프에 대한 예측
        """
        h = self.lin_in(data.x) # (n, d_n) -> (n, d)

        for conv in self.convs:
            h = h + conv(h, data.pos, data.edge_index, data.edge_attr) # (n, d) -> (n, d)
            # 각 MPNN 레이어 뒤에 잔차 연결(residual connection)을 추가하는 점에 주목하세요

        h_graph = self.pool(h, data.batch) # (n, d) -> (batch_size, d)

        out = self.lin_pred(h_graph) # (batch_size, d) -> (batch_size, 1)

        return out.view(-1)

훌륭합니다! 이제 여러분은 더 기하학적으로 원리에 충실한 메시지 패싱 레이어를 정의했고, 이를 사용하여 3D 회전 및 평행이동에 대해 불변인 MPNN 모델을 구성했습니다.

## 💻**Task 3.2:** 새로운 `InvariantMPNNLayer`의 갱신 방정식(update equation)을 적고, 이를 사용하여 해당 레이어와 모델이 3D 회전 및 평행이동에 대해 불변임을 증명하세요. **(1 Mark)**


---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

좋습니다! 여러분은 새로운 `InvariantMPNNLayer`의 갱신 방정식을 성공적으로 작성했고, 그것이 실제로 3D 회전 및 평행이동에 불변임을 보였습니다.

이를 검증하기 위해 몇 가지 정상성 검사를 수행해 봅시다.

## 💻**Task 3.3:** `InvariantMPNNLayer`와 `InvariantMPNNModel`에 대해 단위 테스트를 수행하세요. 그 레이어와 모델이 모두 3D 회전 및 평행이동에 불변임을 보이세요. **(0.5 Mark)**

🤔 *힌트: 앞서 정의한 단위 테스트를 실행하세요.*

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 단위 테스트를 위한 임시 모델, 레이어, 데이터로더를 인스턴스화하세요.
# 이제 InvariantMPNNLayer로 구성된 InvariantMPNNModel을
# 단위 테스트하고 있다는 점을 기억하세요.
#
# layer = ...
# model = ...
# ==========================================
dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# MPNN 모델에 대한 회전 및 평행이동 불변성 단위 테스트
print(f"Is {type(model).__name__} rotation and translation invariant? --> {rot_trans_invariance_unit_test(model, dataloader)}!")

# MPNN 레이어에 대한 회전 및 평행이동 불변성 단위 테스트
print(f"Is {type(layer).__name__} rotation and translation invariant? --> {rot_trans_invariance_unit_test(layer, dataloader)}!")

잘했습니다! 여러분은 `InvariantMPNNLayer`와 `InvariantMPNNModel`을 정의했고, 그 후 그것들이 3D 회전 및 평행이동에 불변임을 증명하고 실험적으로 검증했습니다.

드디어 우리의 기하학적으로 원칙적인 모델로 실험을 실행할 시간입니다!

## 💻**Task 3.4:** `InvariantMPNNModel`을 학습하고 평가하세요. 추가로, 앞서 정의한 기본 `MPNNModel` 및 순진한 `CoordMPNNModel`과 비교하여 이 모델의 결과를 설명하는 몇 문장을 제시하세요. 새 모델이 더 나은가요? 상당한 폭으로 더 나은가요, 아니면 미미하게 더 나은 정도인가요? **(0.5 Mark)**

우리가 제공한 실험 루프를 재사용하고 `YOUR CODE HERE` 섹션을 채워 실험을 실행하세요.

이전의 바닐라 `MPNNModel` 및 순진한 `CoordMPNNModel`과 공정하게 결과를 비교하기 위해, 은닉 차원(hidden dimension) 64로 4개의 메시지 패싱 레이어로 구성된 `InvariantMPNNModel`을 학습하게 됩니다.

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 적절한 인자로 InvariantMPNNModel을 인스턴스화하세요.
#
# model = InvariantMPNNModel(...)
# ==========================================

model_name = type(model).__name__
best_val_error, test_error, train_time, perf_per_epoch = run_experiment(
    model,
    model_name, # "MPNN w/ Features and Coordinates (Invariant Layers)",
    train_loader,
    val_loader,
    test_loader,
    n_epochs=100
)

RESULTS[model_name] = (best_val_error, test_error, train_time)
df_temp = pd.DataFrame(perf_per_epoch, columns=["Test MAE", "Val MAE", "Epoch", "Model"])
DF_RESULTS = DF_RESULTS.append(df_temp, ignore_index=True)

In [ ]:
RESULTS

In [ ]:
p = sns.lineplot(x="Epoch", y="Val MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 2));

In [ ]:
p = sns.lineplot(x="Epoch", y="Test MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 1));

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

멋집니다! 이제 여러분은 바닐라 `MPNNModel`에서 시작하여, `CoordMPNNModel`에서의 순진한 좌표 정보 사용을 거쳐, `InvariantMPNN` 모델에서의 기하학적으로 더 원칙적인 접근에 이르렀습니다.

다음 파트에서는 분자의 기하 구조로부터 얼마나 많은 정보를 끌어낼 수 있는지 그 한계를 더욱 밀어붙여 보겠습니다!

---
---
---

# 🚀 Part 4: 3D 회전 및 평행이동에 대한 등변성(Equivariance)을 갖는 메시지 패싱

이 실습의 이전 파트에서 우리는 **3D 회전** 및 **평행이동** 불변성의 개념을 공부했습니다. 이제 한 걸음 더 나아가겠습니다. **3D 회전 및 평행이동에 등변(equivariant)**인 메시지 패싱 레이어로 구성된, 분자 특성 예측을 위한 GNN을 살펴보겠습니다.

하지만 왜...라고 물을 수 있습니다. 한 발 물러서 봅시다.

### 왜 불변성보다 기하학적 등변성인가?

기하학적 등변성과 대칭성의 필요성을 동기 부여하기 위해, 그래프에 대한 GNN에서의 순열 대칭성(permutation symmetry)과 2D 이미지에 대한 ConvNet에서의 평행이동 대칭성 개념으로 여러분을 다시 데려가고자 합니다.

#### GNN의 순열 대칭성 vs. DeepSets

실습 앞부분에서 우리는 **순열 불변성**과 **등변성** 개념을 복습했습니다. 근본적으로, GNN 레이어는 그래프 노드에 대한 순열 <ins>등변(equivariant)</ins> 연산이어야 합니다. 즉, 그래프의 노드 순서를 바꾸면 레이어의 노드 출력에도 동일한 순열이 적용됩니다. 그러나 그래프 수준 특성 예측을 위한 전체 GNN 모델은 여전히 그래프 노드에 대한 **순열 <ins>불변(invariant)</ins>** 함수입니다. 즉, 노드 순서를 바꾸어도 예측된 그래프 특성은 영향을 받지 않습니다.

강의에서 **[DeepSets 모델](https://arxiv.org/abs/1703.06114)**이 노드 집합에 대한 또 다른 순열 <ins>불변</ins> 아키텍처이며, (방금 말했듯이 역시 순열 불변인) 그래프 수준 특성을 예측하는 데 완벽하게 합리적인 선택지임을 떠올려 보세요. 이는 중요한 질문을 제기합니다: **우리는 왜 순열 <ins>등변</ins> GNN 레이어로 구성된 순열 <ins>불변</ins> GNN 모델을 만들었을까요?**

그 답은, 순열 <ins>등변</ins> GNN 레이어가 모델로 하여금 기저 노드들의 **관계 구조(relational structure)**를 더 잘 활용하게 하고, 이러한 순열 <ins>등변</ins> 연산을 **여러 레이어로 쌓음**으로써 더 강력한 노드 표현(representation)을 구성할 수 있게 하기 때문입니다. (직접 QM9에 대해 DeepSets 모델을 실행해 보고 성능이 떨어지는 것을 확인해 볼 수 있습니다.)

이제 분자 특성 예측 모델에 대한 3D 회전 및 평행이동 대칭성에 동일한 유추를 적용해 봅시다. 지금까지의 여러분의 `InvariantMPNNModel`을 생각해 보세요 — 이는 단지 3D 회전 및 평행이동에 <ins>불변</ins>인 `InvariantMPNNLayer`로 구성되어 있습니다.

순열 <ins>등변</ins> 레이어가 GNN으로 하여금 관계 구조를 더 원칙적인 방식으로 활용하게 했던 것과 유사하게, **3D 회전** 및 **평행이동 <ins>등변</ins> 레이어**는 여러분의 모델이 **기하 구조(geometric structure)**도 더 원칙적인 방식으로 **활용**하게 할 수 있습니다.

#### 2D 이미지용 ConvNet에서의 평행이동 대칭성

<ins>불변</ins> 모델이 <ins>등변</ins> 레이어로 구성되는 또 다른 예는 어디에나 있는 2D 이미지용 **합성곱 신경망(Convolutional Neural Network)**입니다.

ConvNet 모델은 **평행이동**에 <ins>불변</ins>입니다. 즉, 고양이가 이미지의 어디에 위치하든 상관없이 이미지에서 고양이를 검출합니다.

중요한 점은, ConvNet이 입력 이미지 위로 직사각형 창(window)을 슬라이딩하는 것과 유사한 **합성곱 필터(convolution filter)**로 구성된다는 것입니다. 합성곱 필터는 이미지 내의 저수준 패턴을 매칭합니다. 직관적으로, 이러한 필터 중 하나는 고양이 검출 필터일 수 있어, 고양이 같은 픽셀을 마주칠 때마다 발화(fire)합니다. 따라서 합성곱 필터는 출력이 입력과 함께 평행이동하므로 평행이동 <ins>등변</ins> 함수입니다.

<!-- <img src="https://drive.google.com/uc?id=1vgTAG_n5r3H2nqo5vaZPyC60hbZMTEkN" width="100%"> -->
<img src="https://github.com/chaitjo/dump/raw/main/symmetry.png">

([출처](https://bernhard-kainz.com/))

평행이동 <ins>불변</ins> ConvNet은 여러 레이어에 걸쳐 **계층적 특징(hierarchical features)**을 구축하기 위해 평행이동 <ins>등변</ins> 합성곱 필터로 구성됩니다. 깊은 ConvNet을 쌓으면 레이어 간 특징들이 **합성적(compositional)** 방식으로 상호작용할 수 있게 되고, 전체 네트워크가 점점 더 **복잡한 시각적 개념**을 학습할 수 있게 됩니다.

다음 영상은 합성곱 필터의 **평행이동 등변성**에 대한 또 다른 시연을 보여 줍니다: 입력 이미지의 이동이 출력 특징의 이동에 직접 대응됩니다.

([출처](https://fabianfuchsml.github.io/equivariance1of2/))


In [ ]:
HTML('<iframe width="560" height="315" src="https://edwag.github.io/video/translation_equivariance.mp4" allowfullscreen></iframe>')

### 형식화(Formalism)

바라건대, 우리가 3D 회전 및 평행이동 등변 GNN 레이어의 필요성을 충분히 동기 부여했기를 바랍니다. 이제 행렬 표기법을 통해 3D 회전 및 평행이동에 대한 등변성 개념을 형식화해 봅시다.

- $\mathbf{H} \in \mathbb{R}^{n \times d}$를 주어진 분자 그래프의 노드 특징 행렬이라 하자. 여기서 $n$은 노드/원자의 수이고 각 행 $h_i$는 노드 $i$에 대한 $d$차원 특징이다.
- $\mathbf{X} \in \mathbb{R}^{n \times 3}$를 주어진 분자 그래프의 노드 좌표 행렬이라 하자. 여기서 $n$은 노드/원자의 수이고 각 행 $x_i$는 노드 $i$에 대한 3D 좌표이다.
- $\mathbf{A} \in \mathbb{R}^{n \times n}$를 인접 행렬(adjacency matrix)이라 하자. 각 원소 $a_{ij}$는 노드 $i$와 $j$ 사이 간선(edge)의 존재 여부를 나타낸다.
- $\mathbf{F}(\mathbf{H}, \mathbf{X}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times 3} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}^{n \times d}\times \mathbb{R}^{n \times 3}$를 노드 특징, 노드 좌표, 인접 행렬을 입력으로 받아 **갱신된 노드 특징**과 **갱신된 노드 좌표**를 반환하는 **GNN <ins>레이어**</ins>라 하자.
- $f(\mathbf{H}, \mathbf{X}, \mathbf{A}): \mathbb{R}^{n \times d} \times \mathbb{R}^{n \times 3} \times \mathbb{R}^{n \times n} \rightarrow \mathbb{R}$를 노드 특징, 노드 좌표, 인접 행렬을 입력으로 받아 **예측된 그래프 수준 특성**을 반환하는 **GNN <ins>모델**</ins>이라 하자.

우리의 GNN <ins>모델</ins> $f(\mathbf{H}, \mathbf{X}, \mathbf{A})$는 여러 개의 회전 및 평행이동 등변 GNN <ins>레이어</ins> $\mathbf{F}^{\ell}(\mathbf{H}^{\ell}, \mathbf{X}^{\ell}, \mathbf{A}), \ell = 1, 2, \dots, L$로 구성된다.

### 이것은 기하학적으로 불변인 메시지 패싱과 어떻게 다른가?

중요한 점으로, 그리고 회전 및 평행이동 불변 메시지 패싱 레이어와 대조적으로, 등변 메시지 패싱의 각 라운드는 **노드 특징**과 **노드 좌표**를 모두 갱신한다:
$$
\mathbf{H}^{\ell+1}, \mathbf{X}^{\ell+1} = \mathbf{F}^{\ell} (\mathbf{H}^{\ell}, \mathbf{X}^{\ell}, \mathbf{A}).
$$

이러한 형식화는 우리가 **동역학계(dynamical system)**를 모델링하고 있고 노드 좌표가 예컨대 **분자 간 힘(intermolecular forces)**의 작용으로 인해 지속적으로 갱신되고 있다고 믿을 만한 이유가 있는 상황에서, GNN이 유용한 노드 특징을 학습하는 데 매우 유익하다.

기하학적으로 등변인 메시지 패싱 레이어 $\mathbf{F}$에 대한 다음의 미묘한 점들에 유의하라:
- 갱신된 **노드 좌표** $\mathbf{X'}$는 입력 좌표 $\mathbf{X}$의 3D 회전 및 평행이동에 **등변**이다.
- 갱신된 **노드 특징** $\mathbf{H'}$는 (기하학적으로 불변인 메시지 패싱 레이어와 유사하게) 여전히 입력 좌표 $\mathbf{X}$의 3D 회전 및 평행이동에 **불변**이다.
- 전체 **MPNN 모델** $f$는 여전히 3D 회전 및 평행이동에 **불변**일 것이다. 이는 우리가 분자당 **단일 스칼라 양**(전기 쌍극자 모멘트)을 예측하고 있고, 이 값은 원자 좌표의 어떤 회전 및 평행이동 하에서도 변하지 않기 때문이다. 따라서 $L$개의 메시지 패싱 레이어 이후의 최종 노드 특징 벡터들은 그래프 임베딩으로 집계된다(그리고 최종 노드 좌표는 무시된다). 그런 다음 이 그래프 임베딩이 타깃을 예측하는 데 사용된다.

다음 그림은 기하학적으로 불변인 GNN $\mathbf{f}$를 구성하는 데 사용되는 기하학적으로 등변인 메시지 패싱 레이어 $\mathbf{F}$에 대한 이러한 미묘한 점들을 간결하게 담아내고자 한다:

<img src="https://drive.google.com/uc?id=1rRsjM8AdxiU-uJ7C5t1JDMkC19QKdGPg" width="100%">
<!-- <img src="https://github.com/chaitjo/dump/raw/main/gnn-symmetry.png"> -->

이 파트에서 여러분이 탐구하기를 바라는 것은, 이러한 **3D 대칭성**에 **등변**인 **메시지 패싱 레이어**를 사용하여 3D 회전 및 평행이동에 **불변**인 **GNN 모델**을 어떻게 개선할 수 있는가입니다.

시작해 봅시다!

## 💻**Task 4.1:** GNN <ins>레이어</ins> $\mathbf{F}$가 3D 회전과 평행이동에 대해 등변(equivariant)이라는 것은 무슨 의미인가요? 위의 정의들을 사용하여 이를 _수학적으로_ 표현하세요. **(0.5 Marks)**

🤔 *힌트: 앞서 소개한 순열 불변성(permutation invariance) 및 등변성(equivariance), 그리고 3D 회전과 평행이동 불변성에 대한 형식론(formalism)을 다시 살펴보세요.*

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

## 💻**Task 4.2:** 3D 회전 및 평행이동에 <ins>등변(equivariant)</ins>인 새로운 메시지 패싱 레이어를 설계하세요. **(2.5 Marks)**

🤔 *힌트 1: 3D 회전 및 평행이동에 대한 등변성을 보장하려면, 이제 메시지 패싱 레이어가 노드 특징과 노드 좌표를 모두 갱신해야 합니다. 이는 `message()`, `aggregate()`, `update()` 함수 각각이 노드 특징과 노드 좌표로 이루어진 출력 튜플을 주고받게 됨을 의미합니다.*

🤔 *힌트 2: 노드 좌표 쌍 사이에서 계산할 수 있는 어떤 양들은 좌표를 회전하거나 평행이동해도 변하지 않습니다 — 이것들이 **불변량(invariant quantities)**입니다. 반면, 어떤 양들은 좌표와 함께 회전하거나 평행이동할 수 있습니다 — 이것들이 **등변량(equivariant quantities)**입니다. 우리는 여러분이 노드 특징 갱신을 위한 메시지는 3D 회전 및 평행이동에 <ins>불변</ins>이고, 노드 좌표를 위한 메시지는 같은 변환에 <ins>등변</ins>이 되도록 메시지 패싱을 어떻게 설정할 수 있을지 생각하기를 바랍니다.*

**❗️참고:** 이 과제에는 이를 달성하는 여러 가지 가능한 접근법이 있습니다. PyG에서 구현을 직접 임포트하거나 복사하는 것은 유효한 답으로 인정되지 않습니다.

**❗️참고:** 자명한 해 $\mathbf{X}^{\ell+1} = \mathbf{X}^{\ell}$는 유효한 답으로 인정되지 않습니다. GNN에 대한 일반적인 직관은 각 노드가 이웃으로부터 **정보를 빌려 오는(borrow information)** 방법을 학습한다는 것입니다 — 여기서는 노드 특징 정보뿐만 아니라 노드 좌표 정보에 대해서도 이것이 성립합니다. 따라서 우리는 여러분이 이웃의 노드 좌표로부터 집계하여 노드 좌표를 갱신하는 데 메시지 패싱을 사용하기를 바랍니다. 여기서의 '게임'은 3D 대칭성에 등변이 되도록 좌표 메시지 함수를 어떻게 설계하느냐에 관한 것입니다.

In [ ]:
class EquivariantMPNNLayer(MessagePassing):
    def __init__(self, emb_dim=64, edge_dim=4, aggr='add'):
        """메시지 전달 신경망(Message Passing Neural Network) 레이어

        이 레이어는 3D 회전과 평행이동에 대해 등변(equivariant)입니다.

        Args:
            emb_dim: (int) - 은닉 차원 `d`
            edge_dim: (int) - 엣지 특징 차원 `d_e`
            aggr: (str) - 집계 함수 `\oplus` (sum/mean/max)
        """
        # 집계 함수를 설정합니다
        super().__init__(aggr=aggr)

        self.emb_dim = emb_dim
        self.edge_dim = edge_dim

        # ============ 여기에 코드를 작성하세요 ==============
        # 새 레이어를 구성하는 MLP들을 정의하세요.
        # 최소한 `\psi`와 `\phi`가 필요합니다
        # (단, 그 정의는 이전에 사용했던 것과
        # 다를 수 있습니다).
        #
        # self.mlp_msg = ...  # MLP `\psi`
        # self.mlp_upd = ...  # MLP `\phi`
        # ===========================================

    def forward(self, h, pos, edge_index, edge_attr):
        """
        forward 패스는 한 번의 메시지 전달을 통해 노드 특징 `h`를 갱신합니다.

        Args:
            h: (n, d) - 초기 노드 특징
            pos: (n, 3) - 초기 노드 좌표
            edge_index: (e, 2) - 엣지 쌍 (i, j)
            edge_attr: (e, d_e) - 엣지 특징

        Returns:
            out: [(n, d),(n,3)] - 갱신된 노드 특징
        """
        # ============ 여기에 코드를 작성하세요 ==============
        # `forward()` 함수에 초기 노드 좌표를 나타내는 새 인자
        # `pos`가 추가된 것에 주목하세요. 여러분의 과제는
        # `pos`를 다른 인자들과 함께 `message()` 함수로 전달하도록
        # `propagate()` 함수를 갱신하는 것입니다.
        #
        # out = self.propagate(...)
        # return out
        # ==========================================

    # ============ 여기에 코드를 작성하세요 ==============
    # 레이어가 3D 회전과 평행이동에 대해 등변이 되도록 보장하는
    # 사용자 정의 `message()`, `aggregate()`, `update()` 함수를 작성하세요.
    #
    # def message(self, ...):
    #   ...
    #
    # def aggregate(self, ...):
    #   ...
    #
    # def update(self, ...):
    #   ...
    #
    # ==========================================

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(emb_dim={self.emb_dim}, aggr={self.aggr})')


class FinalMPNNModel(MPNNModel):
    def __init__(self, num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1):
        """그래프 속성 예측을 위한 메시지 전달 신경망(Message Passing Neural Network) 모델

        이 모델은 노드 특징과 좌표를 모두 입력으로 사용하며,
        3D 회전과 평행이동에 대해 불변(invariant)입니다 (구성 요소인 MPNN 레이어들이
        3D 회전과 평행이동에 대해 등변이기 때문입니다).

        Args:
            num_layers: (int) - 메시지 전달 레이어 수 `L`
            emb_dim: (int) - 은닉 차원 `d`
            in_dim: (int) - 초기 노드 특징 차원 `d_n`
            edge_dim: (int) - 엣지 특징 차원 `d_e`
            out_dim: (int) - 출력 차원 (1로 고정)
        """
        super().__init__()

        # 초기 노드 특징에 대한 선형 사영
        # dim: d_n -> d
        self.lin_in = Linear(in_dim, emb_dim)

        # MPNN 레이어 스택
        self.convs = torch.nn.ModuleList()
        for layer in range(num_layers):
            self.convs.append(EquivariantMPNNLayer(emb_dim, edge_dim, aggr='add'))

        # 전역 풀링/리드아웃 함수 `R` (평균 풀링)
        # PyG가 `global_mean_pool()`을 통해 내부 로직을 처리합니다
        self.pool = global_mean_pool

        # 선형 예측 헤드
        # dim: d -> out_dim
        self.lin_pred = Linear(emb_dim, out_dim)

    def forward(self, data):
        """
        Args:
            data: (PyG.Data) - PyG 그래프 배치

        Returns:
            out: (batch_size, out_dim) - 각 그래프에 대한 예측
        """
        h = self.lin_in(data.x) # (n, d_n) -> (n, d)
        pos = data.pos

        for conv in self.convs:
            # 메시지 전달 레이어
            h_update, pos_update = conv(h, pos, data.edge_index, data.edge_attr)

            # 노드 특징 갱신
            h = h + h_update # (n, d) -> (n, d)
            # 각 MPNN 레이어 뒤에 잔차 연결(residual connection)을 추가하는 점에 유의하세요

            # 노드 좌표 갱신
            pos = pos_update # (n, 3) -> (n, 3)

        h_graph = self.pool(h, data.batch) # (n, d) -> (batch_size, d)

        out = self.lin_pred(h_graph) # (batch_size, d) -> (batch_size, 1)

        return out.view(-1)

훌륭합니다! 이제 여러분은 3D 회전과 평행이동에 대해 등변인 새로운 메시지 전달 레이어를 정의했고, 이를 사용하여 분자 속성 예측을 위한 최종 MPNN 모델을 구성했습니다.

## 💻**Task 4.3:** 새로 만든 `EquivariantMPNNLayer`의 갱신 방정식(update equation)을 적고, 이를 사용하여 해당 레이어가 3D 회전과 평행이동에 대해 등변임을 증명하세요. **(1 Mark)**

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

훌륭합니다! 여러분은 새로운 `EquivariantMPNNLayer`의 갱신 방정식을 성공적으로 작성했으며, 이것이 실제로 3D 회전과 평행이동에 대해 equivariant함을 보였습니다.

이제 이를 검증하기 위해 몇 가지 sanity check를 수행해 봅시다.

## 💻**Task 4.4:** 여러분의 `EquivariantMPNNLayer`와 `FinalMPNNModel`에 대한 단위 테스트(unit test)를 수행하세요. 먼저, 해당 레이어의 3D 회전 및 평행이동 equivariance에 대한 단위 테스트를 작성하세요. 그런 다음, 레이어가 3D 회전과 평행이동에 대해 equivariant하다는 것과, 모델이 3D 회전과 평행이동에 대해 invariant하다는 것을 보이세요. **(1 Mark)**


In [ ]:
def rot_trans_equivariance_unit_test(module, dataloader):
    """모듈(GNN 레이어)이 회전 및 평행이동에 대해
    등변(equivariant)인지 확인하기 위한 단위 테스트.
    """
    it = iter(dataloader)
    data = next(it)

    out_1, pos_1 = module(data.x, data.pos, data.edge_index, data.edge_attr)

    Q = random_orthogonal_matrix(dim=3)
    t = torch.rand(3)
    # ============ 여기에 코드를 작성하세요 ==============
    # data에 무작위 회전 + 평행이동을 적용하세요.
    #
    # data.pos = ...
    # ==========================================

    # 회전 + 평행이동된 예제에 대한 forward 패스
    out_2, pos_2 = module(data.x, data.pos, data.edge_index, data.edge_attr)

    # ============ 여기에 코드를 작성하세요 ==============
    # 변환을 적용한 후 출력이 변하는지 확인하세요.
    # return ...
    # ==========================================

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 단위 테스트를 위한 임시 모델, 레이어, 데이터로더를 인스턴스화하세요.
# 이제 우리가 단위 테스트하는 대상은 EquivariantMPNNLayer로
# 구성된 FinalMPNNModel이라는 점을 기억하세요.
#
# layer = ...
# model = ...
# ==========================================
dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# MPNN 모델에 대한 회전 및 평행이동 불변성 단위 테스트
print(f"Is {type(model).__name__} rotation and translation invariant? --> {rot_trans_invariance_unit_test(model, dataloader)}!")

# MPNN 레이어에 대한 회전 및 평행이동 불변성 단위 테스트
print(f"Is {type(layer).__name__} rotation and translation equivariant? --> {rot_trans_equivariance_unit_test(layer, dataloader)}!")

드디어! 여러분은 `EquivariantMPNNLayer`와 `FinalMPNNModel`을 정의했고, 그 후 새로운 레이어가 3D 회전과 평행이동에 대해 equivariant함을 증명하고 실험적으로 검증했습니다.

이제 마침내 기하학적으로 원리에 입각한 우리의 최종 모델로 실험을 수행할 시간입니다!

## 💻**Task 4.5:** 여러분의 `FinalMPNNModel`을 학습하고 평가하세요. 추가로, 앞서 정의한 기본 `MPNNModel`, 단순한(naive) `CoordMPNNModel`, 그리고 `InvariantMPNNModel`과 비교하여 이 모델의 결과를 설명하는 몇 문장을 작성하세요. 새로운 모델이 더 나은가요? 상당한 차이로 나은가요, 아니면 조금만 나은가요? **(0.5 Mark)**

제공된 실험 루프를 재사용하고 `YOUR CODE HERE` 부분을 채워서 실험을 실행하세요.

이전의 평범한(vanilla) `MPNNModel`, 단순한 `CoordMPNNModel`, 그리고 `InvariantMPNNModel`과 결과를 공정하게 비교하기 위해, 은닉 차원(hidden dimension)이 64인 4개의 message passing 레이어로 구성된 `EquivariantMPNNModel`을 학습하게 됩니다.

In [ ]:
# ============ 여기에 코드를 작성하세요 ==============
# 적절한 인자로 FinalMPNNModel을 인스턴스화하세요.
#
# model = FinalMPNNModel(...)
# ==========================================

model_name = type(model).__name__
best_val_error, test_error, train_time, perf_per_epoch = run_experiment(
    model,
    model_name, # "MPNN w/ Features and Coordinates (Equivariant Layers)",
    train_loader,
    val_loader,
    test_loader,
    n_epochs=100
)

RESULTS[model_name] = (best_val_error, test_error, train_time)
df_temp = pd.DataFrame(perf_per_epoch, columns=["Test MAE", "Val MAE", "Epoch", "Model"])
DF_RESULTS = DF_RESULTS.append(df_temp, ignore_index=True)

In [ ]:
RESULTS

In [ ]:
p = sns.lineplot(x="Epoch", y="Val MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 2));

In [ ]:
p = sns.lineplot(x="Epoch", y="Test MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 1));

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

축하합니다! 여러분은 이제 평범한 `MPNNModel`에서 출발해, 좌표 정보를 단순하게 사용한 `CoordMPNNModel`을 거쳐, 기하학적으로 더 원리에 입각한 접근인 `InvariantMPNNModel`에 이르렀고, 마침내 3D 회전과 평행이동에 대해 **invariant한 GNN**이면서 이러한 3D 대칭에 대해 **equivariant한 message passing 레이어**로 구성된 `FinalMPNNModel`에 도달했습니다.

다음 파트들에서는 두 가지 서로 다른 설정 하에서 이 모델들을 비교할 것입니다.

---
---
---

# 🌯 Part 5: 마무리

이 섹션에서는 지금까지 살펴본 모델들의 두 가지 중요한 측면, 즉 **샘플 효율성(sample efficiency)**과 **그래프 구조(graph structure)**의 선택을 분석하며 실습을 마무리합니다.

❗️**Note:** 이상적으로는, 이 파트의 과제들에 대해 **새로운 코드를 작성할 필요가 없습니다.** 노트북의 셀들을 실행하고 보이는 실험 결과를 추론하기만 하면 됩니다. 이것은 여러분이 자신의 연구 논문을 읽거나 쓸 때 표와 그림을 어떻게 해석해야 하는지를 시뮬레이션하는 연습입니다.

### 샘플 효율성 (Sample Efficiency)

먼저 샘플 효율성에 대해 생각해 보길 바랍니다 — 모델 A가 모델 B보다 샘플 효율적이라는 것은, 모든 샘플에서 최대한을 끌어낼 수 있다는 의미, 즉 더 적은 데이터로 더 나은 성능에 도달할 수 있다는 뜻입니다.

## 💻**Task 5.1:** 학습 에폭(epoch) 수에 따른 모든 모델의 성능을 분석하세요. 무엇이 관찰되나요? 여러분의 발견을 설명하세요. (1 Mark)

**학습 에폭** 수를 **학습 샘플** 수에 대한 대용 지표(proxy)로 간주할 수 있습니다. 즉, 모델이 더 적은 에폭 안에 더 나은 성능으로 수렴한다면 그 모델은 더 샘플 효율적인 것입니다.

학습 샘플 수에 따른 모델들의 성능을 비교하세요. 표준 `MPNNModel`, `CoordMPNNModel`, `InvariantMPNNModel`, 그리고 `FinalMPNN`의 서로 다른 모델링 가정들이 샘플 효율성에 어떤 영향을 미치나요? 어떤 모델이 저샘플(low-sample) 영역에서 가장 좋은 성능을 보이나요? 샘플 크기를 늘리면 어떤 일이 일어나나요?

이 질문에 답하기 위해, 제공된 `sns.lineplot()` 함수와 `DF_RESULTS`의 결과를 사용하여 학습 에폭 수에 대한 검증 및 테스트 세트 MAE를 시각화하세요.

**❗️Note:** 실습에서 모든 모델을 성공적으로 구현하지 못했더라도 이 과제를 시도하는 것을 강력히 권장합니다. 여러분이 이해하고 구현한 모델들만을 바탕으로 답하면 됩니다!

In [ ]:
p = sns.lineplot(x="Epoch", y="Val MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 2));

In [ ]:
p = sns.lineplot(x="Epoch", y="Test MAE", hue="Model", data=DF_RESULTS)
p.set(ylim=(0, 1));

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

### 밀집(Dense) vs. 희소(Sparse) 그래프

이제 **기반이 되는 그래프 구조**의 선택으로 관심을 돌려 봅시다.

이 실습에서 우리는 분자를 표현하기 위해 완전 연결(fully-connected) 인접 행렬을 사용해 왔습니다(즉, 분자 내 모든 원자가 자기 자신을 제외하고 서로 연결됨). 그러나 분자 그래프에 대한 정보는 항상 엣지 속성 `data.edge_attr`을 통해 모델에 제공되어 왔다는 점에 유의하세요:
- 두 원자가 물리적으로 연결되어 있을 때, 엣지 속성은 결합 유형(단일, 이중, 삼중, 또는 방향족)을 원-핫(one-hot) 벡터로 나타냅니다.
- 두 원자가 물리적으로 연결되어 있지 **않을** 때는, 모든 엣지 속성이 0입니다.

다음 과제에서는 완전 연결 인접 행렬과 희소(sparse) 인접 행렬(두 원자 사이에 물리적 연결이 존재할 때만 엣지가 존재하는 경우)의 장점/단점을 살펴볼 것입니다.

## 💻**Task 5.2:** 두 가지 시나리오(완전 연결 vs. 희소 그래프)에서 모델들의 성능을 비교하세요. 여러분의 발견을 설명하세요. (1 Mark)

희소 형식으로 데이터셋을 불러오는 코드가 제공됩니다. 모든 모델이 희소 형식으로 학습을 마칠 때까지 다소 기다려야 할 수도 있습니다.

커피나 차 한 잔 하세요! ☕️

**❗️Note:** 다시 한번, 실습에서 모든 모델을 성공적으로 구현하지 못했더라도 이 과제를 시도하는 것을 강력히 권장합니다. 여러분이 이해하고 구현한 모델들만을 바탕으로 답하면 됩니다!

In [ ]:
# 희소 그래프로 QM9 데이터셋 로드 (full graphs 변환을 제거하여)
sparse_dataset = QM9(path, transform=SetTarget())

# 각 데이터 샘플별로 타깃을 평균 = 0, 표준편차 = 1이 되도록 정규화.
mean = sparse_dataset.data.y.mean(dim=0, keepdim=True)
std = sparse_dataset.data.y.std(dim=0, keepdim=True)
sparse_dataset.data.y = (sparse_dataset.data.y - mean) / std
mean, std = mean[:, target].item(), std[:, target].item()

# 데이터셋 분할 (3K 부분집합)
train_dataset_sparse = sparse_dataset[:1000]
val_dataset_sparse = sparse_dataset[1000:2000]
test_dataset_sparse = sparse_dataset[2000:]
print(f"Created sparse dataset splits with {len(train_dataset_sparse)} training, {len(val_dataset_sparse)} validation, {len(test_dataset_sparse)} test samples.")

# 배치 크기 = 32로 데이터로더 생성
train_loader_sparse = DataLoader(train_dataset_sparse, batch_size=32, shuffle=True)
test_loader_sparse = DataLoader(test_dataset_sparse, batch_size=32, shuffle=False)
val_loader_sparse = DataLoader(val_dataset_sparse, batch_size=32, shuffle=False)

이제 희소 데이터셋이 실습 전반에 걸쳐 사용해 온 완전 연결 데이터셋보다 실제로 더 희소한지 확인해 봅시다:

In [ ]:
val_batch_sparse = next(iter(val_loader_sparse))
val_batch_dense = next(iter(val_loader))

# 이 두 배치는 동일한 분자에 대응해야 합니다. sanity check를 추가해 봅시다
assert torch.allclose(val_batch_sparse.y, val_batch_dense.y, atol=1e-4)

print(f"Number of edges in sparse batch {val_batch_sparse.edge_index.shape[-1]}. Number of edges in dense batch {val_batch_dense.edge_index.shape[-1]}")

이제 두 시나리오에서 모델들을 비교해 봅시다:

In [ ]:
sparse_results = {}
dense_results = RESULTS

In [ ]:
# ============ YOUR CODE HERE ==============
# 모델들을 인스턴스화
models = [MPNNModel(), CoordMPNNModel(), InvariantMPNNModel(), FinalMPNNModel()]
# ==========================================

for model in models:
  model_name = type(model).__name__

  if model_name not in sparse_results:
    sparse_results[model_name] = run_experiment(
        model,
        model_name,
        train_loader_sparse,
        val_loader_sparse,
        test_loader_sparse,
        n_epochs=100
    )

  if model_name not in dense_results:
    dense_results[model_name] = run_experiment(
        model,
        model_name,
        train_loader,
        val_loader,
        test_loader,
        n_epochs=100
    )

In [ ]:
df_sparse = pd.DataFrame.from_dict(sparse_results, orient='index', columns=['Best val MAE', 'Test MAE', 'Train time', 'Train History'])
df_dense = pd.DataFrame.from_dict(dense_results, orient='index', columns=['Best val MAE', 'Test MAE', 'Train time'])
df_sparse['type'] = 'sparse'
df_dense['type'] = 'dense'
df = df_sparse.append(df_dense)

sns.set(rc={'figure.figsize':(10, 6)})
sns.barplot(x=df.index, y="Test MAE", hue="type", data=df);

# 이 플롯을 저장하고 다운로드하고 싶을 수 있습니다
# plt.savefig("comparison.png")
# files.download("comparison.png")

두 시나리오에서 모델들의 성능을 비교하세요. 어떤 모델이 더 좋게/나쁘게 수행되었나요? 왜 그렇다고 생각하나요? 완전 연결(fully-connected) 시나리오와 희소(sparse) 시나리오 사이에서 어떤 차이를 관찰했나요? 그 차이를 설명하기 위해 최소 *두 가지* 논거를 제시하세요.

---

<font color='red'>❗️여기에 답을 작성하세요</font>

---

---
---
---

[Fin.](https://www.youtube.com/watch?v=b9434BoGkNQ)

---
---
---

# FAQ 및 일반적인 조언

### 단위 테스트 (Unit Testing)
여러분이 생각하는 동작이 실제로 일어나고 있는지 확인하기 위한 sanity check로 단위 테스트 함수를 사용하는 것을 고려해 보세요. 우리가 요청한 것을 넘어서, 레이어/모델의 특정 속성이나 학습 중 발생하는 수치적 문제를 테스트하기 위해 직접 단위 테스트를 작성하는 것도 고려해 볼 수 있습니다.

### 이론 vs. 실험 결과
여러분이 제안한 레이어가 이론적으로 타당하다고 확신한다면, 예를 들어 Task 4.2에서 여러분의 해법이 3D equivariance를 만족하지만 결과가 인상적이지 않거나 안정적인 학습을 달성하지 못한다면, 이는 수치적 불안정성이나 엔지니어링 문제 때문일 수 있습니다. 그러한 경우이고 결국 그 문제들을 극복하지 못하더라도, 여러분이 가진 것을 그대로 해법으로 제출해도 됩니다. 이론이 옳다면 부분 점수를 받게 됩니다.

### Google Colab의 GPU 사용량 제한(rate-limit)
**요약(TL;DR)** 당황하지 말고, 일찍 시작하고, 매번 다시 실행하는 대신 결과를 저장하고 불러오세요.

실습을 테스트하는 동안 우리는 사용량 제한을 여러 번 겪었습니다. 사용자당 12~24시간마다 GPU 연산량에 상한이 있는 것으로 보입니다. 한도에 도달하면 Colab이 GPU 런타임 연결을 끊습니다(다음 날에는 다시 GPU에 연결됩니다). 따라서 우리는 실습을 최대한 연산적으로 단순하게 유지하려고 노력했습니다.

생활을 좀 더 수월하게 만들기 위한 몇 가지 제안이 있으며, 다음 항목들에 나열합니다:
- 가능하다면, 마지막 순간까지 미루지 마세요. 마감일 당일에 사용량 제한과 씨름하지 않도록 일찍 시작하세요!
- 만약 사용량 제한에 걸린다면, 예를 들어 각 100 샘플처럼 아주 작은 데이터셋 크기로 모델 구현을 작성하고 테스트하는 것을 고려할 수 있습니다. GPU에 다시 연결되면, 전체 데이터셋으로 모델을 다시 실행할 수 있습니다.
- 정기적으로 사용량 제한에 걸린다면(예: 다른 Colab 프로젝트도 동시에 실행 중이라면 이런 일이 자주 발생할 수 있습니다), 각 과제의 결과를 Google Drive/로컬 저장소에 저장하고 노트북을 다시 실행할 때마다 단순히 불러오면 됩니다.
- 한 계정에서 사용량 제한에 걸렸다면, 새 Google 계정, 여러분의 cam.ac.uk 계정, 그리고/또는 새 IP 주소를 조합하여 새로운 GPU 런타임을 얻을 수 있습니다.


